# BEH 20: customer activity by month

Everything a customer did, or had done to them, counted per customer and calendar month (`stg.beh_activity_month`,
`stg.beh_comms_month`), ready for the rolling windows in BEH_30.

Sources: linked ads_v2 episode events (flags by event code, `build/beh_codes.py`), episodes, Genesys voice calls linked
by phone number through ads_v2.ContactScvLink, MTA postings (ods_v2.PolicyMtaDay), MTA change detail, finance loan
status changes, new-business sales, MyChill document decisions, marketing consent and campaign history.

In-life (`_il`) counts: activity that belongs to an ads_v2 episode is in-life when the episode is not a renewal
(acquisition, MTA, document requests, arrears, claims, cancellation). Activity without an episode is in-life when its
day falls outside every renewal window of the customer (offered renewal due date minus 60 to plus 30 days).

**If a cell fails**

1. Read the failing cell's header: what it builds, what it reads, its grain and key.
2. Fix the cause (usually a renamed or missing source column, or a broken key), then rerun from that cell.
   Every cell is `create or replace`, so rerunning is safe.
3. The checks cell at the end names any table that is empty or has duplicate / null keys, and the likely cause.

Run order: BEH_00_Setup is run by every notebook. Then 10, 20, 30, 40, 50, 60, in that order.


In [ ]:
%%pyspark
# Parameters (leave blank when running by hand)
batch_id = ""


In [ ]:
%%pyspark
%run BEH_00_Setup

In [ ]:
%%pyspark
beh_run_log("BEH_20_Activity", "started")


## beh_renewal_due


In [ ]:
/****************************************************************************************************/
/*  Name        : stg.beh_renewal_due                                                               */
/*  Description : Every Motor, Home and Van renewal cycle with its customer and due date, offered   */
/*                or not. Gives the renewal windows (offered only) and the renewals-due             */
/*                opportunity counts.                                                               */
/*  Reads       : ods_v2.PolicyRenewal, stg.beh_policy_scv                                          */
/*  Key         : PolicyCode + DueDate                                                              */
/****************************************************************************************************/
create or replace table ${beh.stg}.beh_renewal_due as
select  l.SCV_Customer_Key,
        r.PolicyCode,
        r.LYPolicyRenewDateAdj                                              as DueDate,
        year(r.LYPolicyRenewDateAdj) * 12 + month(r.LYPolicyRenewDateAdj)   as MonthNo,
        r.LYPolicyTypeGroup                                                 as PolicyTypeGroup,
        r.PolicyOfferNum,
        r.PolicyRetNum
from    ${beh.ods}.policyrenewal r
join    ${beh.stg}.beh_policy_scv l on l.PolicyCode = r.PolicyCode
where   r.LYPolicyTypeGroup in ('Motor', 'Home', 'Van')
  and   r.LYPolicyRenewDateAdj >= add_months(to_date('${beh.hist_start}'), -36)
  and   r.LYPolicyRenewDateAdj <= date_add(to_date('${beh.last_me}'), 120)


## beh_event


In [ ]:
/****************************************************************************************************/
/*  Name        : stg.beh_event                                                                     */
/*  Description : Linked episode events in the build window with one 0/1 flag per behaviour (code   */
/*                lists in build/beh_codes.py). NeedsReview is carried, not filtered: ads_v2 sets   */
/*                it on most events for structural reasons (no spine, gap fallback), not for bad    */
/*                data. TimeUsable = 1 when the event time can order events: an actual time, or R0  */
/*                / HR0 / VR0 whose time is set by rule (window opens). Derived times (R1.B2, R3a,  */
/*                R3c, R3d, placed at due minus 60 or at due) cannot. ActivityDate is the date the  */
/*                rolling windows count the event on: the event date, except that events ads_v2     */
/*                places at the renewal start without an actual time                                */
/*                (beh_codes.RENEWAL_START_DERIVED) count at the due date. InLife = 1 for events    */
/*                of a non-renewal episode, 0 for a renewal episode, null without an episode.       */
/*                Channel (online / phone / chat) and is_assisted come from the beh_codes channel   */
/*                lists.                                                                            */
/*  Reads       : ads_v2.EpisodeEventsNumbered, ads_v2.ScvEventStream (EventTimeBasis)              */
/*  Key         : EventId                                                                           */
/****************************************************************************************************/
create or replace table ${beh.stg}.beh_event as
    select  e.EventId,
            e.SCV_Customer_Key,
            e.EventDate,
            e.EventDateTime,
            e.EventTypeId,
            lower(e.StepType)       as StepType,
            lower(e.EpisodeType)    as EpisodeType,
            lower(e.ProductType)    as ProductType,
            e.EpisodeKey,
            e.EffectivePolicyCode,
            e.NeedsReview,
            case when e.EpisodeType is null then null
                 when lower(e.EpisodeType) = 'renewals' then 0 else 1 end
                                    as InLife,
            s.EventTimeBasis,
            case when s.EventTimeBasis = 'actual' or e.EventTypeId in ('R0', 'HR0', 'VR0') then 1 else 0 end
                                    as TimeUsable,
            case when coalesce(s.EventTimeBasis, '') <> 'actual'
                  and e.EventTypeId in ('R3a', 'R4.F1', 'R4a', 'R4c')
                 then date_add(e.EventDate, 60) else e.EventDate end
                                    as ActivityDate,
            case when e.EventTypeId in ('R0', 'R1a', 'R1b', 'R1.E1', 'R1.B2', 'R3.B1.2', 'HR0', 'HR0.1', 'HR1a', 'HR1b', 'HR1.E1', 'HR3.B1.2', 'VR0', 'VR0.1', 'VR1a', 'VR1b', 'VR1.E1', 'VR3.B1.2') then 1 else 0 end as prompts,
case when e.EventTypeId in ('R2', 'R2a', 'R2b', 'R3a', 'R3b', 'R3d', 'HR2', 'HR2a', 'HR2b', 'HR3a', 'HR3b', 'VR2', 'VR2a', 'VR2b', 'VR3a', 'VR3b') then 1 else 0 end as ren_cust_actions,
case when e.EventTypeId in ('M1', 'HM1', 'VM1', 'D1', 'HD1', 'VD1', 'M0.1', 'HM0.1', 'VM0.1') then 1 else 0 end as inlife_contacts,
case when e.EventTypeId in ('A6.B1.4', 'A6.B1.5', 'R4.B2.4', 'R4.B2.5', 'M3.B1.4', 'M3.B1.5', 'C2', 'C3', 'HA6.B1.4', 'HA6.B1.5', 'HR4.B2.4', 'HR4.B2.5', 'HM4.B1.4', 'HM4.B1.5', 'HC2', 'HC3', 'VA6.B1.4', 'VA6.B1.5', 'VR4.B2.4', 'VR4.B2.5', 'VM3.B1.4', 'VM3.B1.5', 'VC2', 'VC3') then 1 else 0 end as chases,
case when e.EventTypeId in ('R2.E1') then 1 else 0 end       as concessions,
case when e.EventTypeId in ('A5.F1', 'R4.F1', 'M4.F1', 'HA5.F1', 'HR4.F1', 'HM4.F1', 'VR4.F1', 'VM4.F1') then 1 else 0 end as failed_payments,
case when e.EventTypeId in ('R2.F1') then 1 else 0 end       as failed_logins,
case when e.EventTypeId in ('ARR.1', 'ARR.2', 'ARR.3', 'HARR.1', 'HARR.2', 'HARR.3', 'VARR.1', 'VARR.2', 'VARR.3') then 1 else 0 end as arrears_events,
case when e.EventTypeId in ('D1', 'HD1', 'VD1') then 1 else 0 end as doc_requests,
case when e.EventTypeId in ('A0', 'HA0', 'VA0') then 1 else 0 end as sales_calls,
case when e.EventTypeId in ('A1') then 1 else 0 end          as online_quotes_motor,
case when e.EventTypeId in ('HA1') then 1 else 0 end         as online_quotes_home,
case when e.EventTypeId in ('VA1') then 1 else 0 end         as online_quotes_van,
case when e.EventTypeId in ('R2.B1.1', 'HR2.B1.1', 'VR2.B1.1') then 1 else 0 end as renewal_requotes,
case when e.EventTypeId in ('R4.B2.F1', 'VR4.B2.F1', 'VM3.B1.F1', 'HR4.B2.F1', 'HA6.F1', 'A6.F1', 'A6.B1.F1', 'VA6.B1.F1', 'A7.F1', 'HM4.B1.F1') then 1 else 0 end as doc_friction_events,
case when e.EventTypeId in ('D3.F1', 'HD3.F1', 'VD3.F1', 'A7.B1.5', 'HA7.B1.5', 'R6.B1.5', 'HR6.B1.5', 'VR6.B1.5') then 1 else 0 end as doc_nonreceipt,
case when lower(e.StepType) = 'friction' and e.EventTypeId not in ('A3.F1', 'R2.B1.4.F1', 'R4.B2.F1', 'VR4.B2.F1', 'VM3.B1.F1', 'HR4.B2.F1', 'HA6.F1', 'A6.F1', 'A6.B1.F1', 'VA6.B1.F1', 'A7.F1', 'HM4.B1.F1', 'HR1.E1.F1', 'HA2.F1', 'HR2.B1.2.F1', 'HA5.F1', 'HR2.B1.F1', 'HA1.F1', 'HR2.B1.3.F1', 'HA3.F1', 'HR2.F1', 'VR3.B1.2.F1', 'R3.B1.2.F1') then 1 else 0 end as friction_events,
case when e.EventTypeId in ('R1a', 'R1b', 'HR1a', 'HR1b', 'VR1a', 'VR1b') then 1 else 0 end as is_invite,
case when e.EventTypeId in ('A0', 'A4b', 'R2b', 'R3c', 'M1', 'D1', 'C1a', 'HA0', 'HA4b', 'HR2b', 'HM1', 'HD1', 'HC1a', 'VA0', 'VR2b', 'VM1', 'VD1', 'VC1a', 'R3d', 'A4.F1', 'HA4.F1', 'A5b') then 1 else 0 end as is_assisted,
case when e.EventTypeId in ('A1', 'A4a', 'A5a', 'R2a', 'R2.F1', 'R3a', 'R2.B1.1', 'HA1', 'HA4a', 'HR2a', 'HR3a', 'HR2.B1.1', 'VA1', 'VR2a', 'VR3a', 'VR2.B1.1') then 'online' when e.EventTypeId in ('A0', 'A4b', 'R2b', 'R3c', 'M1', 'D1', 'C1a', 'HA0', 'HA4b', 'HR2b', 'HM1', 'HD1', 'HC1a', 'VA0', 'VR2b', 'VM1', 'VD1', 'VC1a') then 'phone' when e.EventTypeId in ('R3d') then 'chat' end as Channel
    from    ${beh.ads}.episodeeventsnumbered e
    left join ${beh.ads}.scveventstream s on s.EventId = e.EventId
    where   e.SCV_Customer_Key is not null
      and   e.EventDate >= to_date('${beh.cov_events}')
      and   e.EventDate <= to_date('${beh.last_me}')


## beh_episode


In [ ]:
/****************************************************************************************************/
/*  Name        : stg.beh_episode                                                                   */
/*  Description : One row per numbered episode: customer, journey, product, start and end date,     */
/*                whether it holds a friction event (stable code set, as friction_events) and       */
/*                whether it reached an outcome step. Episodes count in the rolling windows at      */
/*                their end date, when friction and completion are known.                           */
/*  Reads       : stg.beh_event                                                                     */
/*  Key         : EpisodeKey                                                                        */
/****************************************************************************************************/
create or replace table ${beh.stg}.beh_episode as
select  EpisodeKey,
        min(SCV_Customer_Key)                                       as SCV_Customer_Key,
        min(EpisodeType)                                            as EpisodeType,
        min(ProductType)                                            as ProductType,
        min(EventDate)                                              as EpisodeStartDate,
        max(ActivityDate)                                           as EpisodeEndDate,
        count(*)                                                    as Events,
        max(friction_events)                                        as HasFriction,
        max(case when StepType = 'outcome' then 1 else 0 end)       as HasOutcome,
        max(is_assisted)                                            as HasAssistedStep,
        max(InLife)                                                 as InLife,
        max(NeedsReview)                                            as NeedsReview
from    ${beh.stg}.beh_event
where   EpisodeKey is not null
group by EpisodeKey


## beh_call


In [ ]:
/****************************************************************************************************/
/*  Name        : stg.beh_call                                                                      */
/*  Description : One row per Genesys voice conversation: start, direction, the customer's phone    */
/*                key (ani inbound, dnis outbound), wait, talk, abandoned / answered, and whether   */
/*                it touched a QD_ queue (out of scope).                                            */
/*  Reads       : ods_v2.CallSession                                                                */
/*  Key         : ConversationId                                                                    */
/****************************************************************************************************/
create or replace table ${beh.stg}.beh_call as
select  ConversationId,
        min(ConversationStartTime)                                                          as CallStart,
        max(OriginatingDirection)                                                           as Direction,
        max(case when OriginatingDirection = 'inbound' then AniPhoneKey else DnisPhoneKey end) as PhoneKey,
        max(coalesce(Abandoned, 0))                                                         as Abandoned,
        max(coalesce(AgentAnswered, 0))                                                     as AgentAnswered,
        max(coalesce(AcdWaitDurationMs, 0))                                                 as WaitMs,
        sum(coalesce(AgentTalkDurationMs, 0))                                               as TalkMs,
        max(case when substr(upper(QueueName), 1, 3) = 'QD_' then 1 else 0 end)             as IsQdQueue
from    ${beh.ods}.callsession
where   MediaType = 'voice'
  and   ConversationStartTime >= to_timestamp('${beh.cov_calls}')
  and   ConversationStartTime < date_add(to_date('${beh.last_me}'), 1)
group by ConversationId


## beh_call_scv


In [ ]:
/****************************************************************************************************/
/*  Name        : stg.beh_call_scv                                                                  */
/*  Description : Voice conversations linked to exactly one SCV customer: the phone key's contact   */
/*                link valid at the call start. Calls whose number links to several customers at    */
/*                that time are left out (not guessed); QD_ queues are out of scope.                */
/*  Reads       : stg.beh_call, ads_v2.ContactScvLink (PhoneKey)                                    */
/*  Key         : ConversationId                                                                    */
/****************************************************************************************************/
create or replace table ${beh.stg}.beh_call_scv as
select  k.ConversationId,
        min(l.SCV_Customer_Key)     as SCV_Customer_Key,
        k.CallStart,
        to_date(k.CallStart)        as CallDate,
        k.Direction,
        k.Abandoned,
        k.AgentAnswered,
        k.WaitMs,
        k.TalkMs
from    ${beh.stg}.beh_call k
join    ${beh.ads}.contactscvlink l
        on  l.ContactKeyType = 'PhoneKey'
        and l.ContactKey = k.PhoneKey
        and k.CallStart >= l.ValidFrom
        and (l.ValidTo is null or k.CallStart < l.ValidTo)
where   k.IsQdQueue = 0
  and   k.PhoneKey is not null
group by k.ConversationId, k.CallStart, k.Direction, k.Abandoned, k.AgentAnswered, k.WaitMs, k.TalkMs
having  count(distinct l.SCV_Customer_Key) = 1


## beh_event_channel


In [ ]:
/****************************************************************************************************/
/*  Name        : stg.beh_event_channel                                                             */
/*  Description : Episode steps with a channel (online, phone, chat) and a usable time, each with   */
/*                the channel of the step before it in the same episode. Derived-time steps are     */
/*                left out, so switches are a lower bound (curation #26).                           */
/*  Reads       : stg.beh_event                                                                     */
/*  Key         : EventId                                                                           */
/****************************************************************************************************/
create or replace table ${beh.stg}.beh_event_channel as
select  EventId,
        EpisodeKey,
        EventDateTime,
        Channel,
        lag(Channel) over (partition by EpisodeKey order by EventDateTime, EventId)  as PrevChannel
from    ${beh.stg}.beh_event
where   EpisodeKey is not null
  and   Channel is not null
  and   TimeUsable = 1


## beh_episode_call


In [ ]:
/****************************************************************************************************/
/*  Name        : stg.beh_episode_call                                                              */
/*  Description : Linked voice calls placed in the customer's episode whose first to last event     */
/*                spans the call date. A call inside several episodes goes to the one that started  */
/*                last.                                                                             */
/*  Reads       : stg.beh_call_scv, stg.beh_episode                                                 */
/*  Key         : ConversationId                                                                    */
/****************************************************************************************************/
create or replace table ${beh.stg}.beh_episode_call as
select  ConversationId,
        EpisodeKey,
        Direction
from    (select  c.ConversationId,
                 p.EpisodeKey,
                 c.Direction,
                 row_number() over (partition by c.ConversationId
                                    order by p.EpisodeStartDate desc, p.EpisodeKey desc) as rn
         from    ${beh.stg}.beh_call_scv c
         join    ${beh.stg}.beh_episode p
                 on  p.SCV_Customer_Key = c.SCV_Customer_Key
                 and c.CallDate between p.EpisodeStartDate and p.EpisodeEndDate) k
where   rn = 1


## beh_episode_x


In [ ]:
/****************************************************************************************************/
/*  Name        : stg.beh_episode_x                                                                 */
/*  Description : Episodes with the call and channel readings: HasAssisted (an agent step or a      */
/*                linked call in the span, #25), RepeatContact (two or more linked inbound calls    */
/*                in the span, #43), ChannelSwitches and HasChannelStep (#26).                      */
/*  Reads       : stg.beh_episode, stg.beh_episode_call, stg.beh_event_channel                      */
/*  Key         : EpisodeKey                                                                        */
/****************************************************************************************************/
create or replace table ${beh.stg}.beh_episode_x as
select  p.*,
        case when p.HasAssistedStep = 1 or coalesce(c.Calls, 0) > 0 then 1 else 0 end  as HasAssisted,
        case when coalesce(c.InboundCalls, 0) >= 2 then 1 else 0 end                   as RepeatContact,
        coalesce(c.InboundCalls, 0)                                                    as InboundCalls,
        case when h.EpisodeKey is not null then 1 else 0 end                           as HasChannelStep,
        coalesce(h.ChannelSwitches, 0)                                                 as ChannelSwitches
from    ${beh.stg}.beh_episode p
left join (select  EpisodeKey,
                   count(*)                                                   as Calls,
                   sum(case when Direction = 'inbound' then 1 else 0 end)     as InboundCalls
           from    ${beh.stg}.beh_episode_call
           group by EpisodeKey) c on c.EpisodeKey = p.EpisodeKey
left join (select  EpisodeKey,
                   sum(case when PrevChannel is not null and Channel <> PrevChannel then 1 else 0 end) as ChannelSwitches
           from    ${beh.stg}.beh_event_channel
           group by EpisodeKey) h on h.EpisodeKey = p.EpisodeKey


## beh_mta


In [ ]:
/****************************************************************************************************/
/*  Name        : stg.beh_mta                                                                       */
/*  Description : One row per mid-term adjustment (history types P permanent and T temporary; one   */
/*                per policy and history sequence), dated when processed, with forced (vehicle,     */
/*                registration, address) and elective (cover, excess, NCD, drivers, use, add-ons)   */
/*                flags and change categories.                                                      */
/*  Reads       : ods_v2.PolicyMta, stg.beh_policy_scv                                              */
/*  Key         : PolicyCode + HistorySeq                                                           */
/****************************************************************************************************/
create or replace table ${beh.stg}.beh_mta as
select  l.SCV_Customer_Key,
        m.PolicyCode,
        m.HistorySeq,
        min(m.ProcessedDate)                                                                         as ActivityDate,
        max(case when 'Y' in (m.ChangeAddress, m.PermanentSubstitution, m.ChangeRegistrationFlag,
                              m.AmendVehicleDetails) then 1 else 0 end)                              as IsForced,
        max(case when 'Y' in (m.ChangeOfCover, m.ChangePolicyExcess, m.ChangeNcdProtection, m.ChangeNcd,
                              m.PermanentAdditionalDrivers, m.ChangeOfAdditionalDriverDetails,
                              m.ChangeDrivingOption, m.ChangeClassOfUse, m.ChangeAreaOfUse,
                              m.Amend2ndCarDiscount, m.AmendAdditionalItems) then 1 else 0 end)       as IsElective,
        max(case when 'Y' in (m.ChangeOfCover, m.ChangePolicyExcess, m.ChangeNcdProtection, m.ChangeNcd)
                 then 1 else 0 end)                                                                  as CatCover,
        max(case when 'Y' in (m.PermanentAdditionalDrivers, m.ChangeOfAdditionalDriverDetails,
                              m.ChangeDrivingOption) then 1 else 0 end)                              as CatDrivers,
        max(case when 'Y' in (m.PermanentSubstitution, m.ChangeRegistrationFlag, m.AmendVehicleDetails)
                 then 1 else 0 end)                                                                  as CatVehicle,
        max(case when m.ChangeAddress = 'Y' then 1 else 0 end)                                       as CatAddress,
        max(case when 'Y' in (m.AmendLicence, m.AmendLicenceDetails, m.AmendConvictionsProposer,
                              m.AmendConvictionsOther) then 1 else 0 end)                            as CatLicence,
        max(case when 'Y' in (m.ChangeClassOfUse, m.ChangeAreaOfUse) then 1 else 0 end)              as CatUse,
        max(case when m.ChangeOfPolicyholderDetails = 'Y' then 1 else 0 end)                         as CatPersonal
from    ${beh.ods}.policymta m
join    ${beh.stg}.beh_policy_scv l on l.PolicyCode = m.PolicyCode
where   m.HistoryType in ('P', 'T')
  and   m.ProcessedDate >= to_date('${beh.cov_mta}')
  and   m.ProcessedDate <= to_date('${beh.last_me}')
group by l.SCV_Customer_Key, m.PolicyCode, m.HistorySeq


## beh_pme_mta


In [ ]:
/****************************************************************************************************/
/*  Name        : stg.beh_pme_mta                                                                   */
/*  Description : MTA counts per policy and posting day with the warehouse rule                     */
/*                (ods_v2.PolicyMtaDay.MTAGrossNum: each MTA posting +1, a reversal -1, summing to  */
/*                edw.tbl_fact_policy_mvt.MTAGrossNum per month). Full history from 2021-04, dated  */
/*                to the day, so in-life counts can be split. ods_v2.PolicyMta only holds history   */
/*                from 2026-01-22, so it gives the change types, not the counts.                    */
/*  Reads       : ods_v2.PolicyMtaDay, ods_v2.Policy, stg.beh_policy_scv                            */
/*  Key         : SCV_Customer_Key + PolicyCode + ActivityDate                                      */
/****************************************************************************************************/
create or replace table ${beh.stg}.beh_pme_mta as
select  l.SCV_Customer_Key,
        d.PolicyCode,
        d.WorkDate                      as ActivityDate,
        cast(d.MTAGrossNum as int)      as MtaCount
from    ${beh.ods}.policymtaday d
join    ${beh.ods}.policy p on p.PolicyCode = d.PolicyCode
join    ${beh.stg}.beh_policy_scv l on l.PolicyCode = d.PolicyCode
where   p.PolicyTypeGroup in ('Motor', 'Home', 'Van')
  and   d.MTAGrossNum <> 0
  and   d.WorkDate >= to_date('${beh.cov_mta_count}')
  and   d.WorkDate <= to_date('${beh.last_me}')


## beh_loan_change


In [ ]:
/****************************************************************************************************/
/*  Name        : stg.beh_loan_change                                                               */
/*  Description : Finance loan status changes (a status run whose status differs from the previous  */
/*                one); the opening status of a loan is not a change.                               */
/*  Reads       : ods_v2.PolicyLoanStatus, stg.beh_policy_scv                                       */
/*  Key         : LoanReference + ActivityDate                                                      */
/****************************************************************************************************/
create or replace table ${beh.stg}.beh_loan_change as
select  l.SCV_Customer_Key,
        s.LoanReference,
        s.StatusFromDate        as ActivityDate,
        s.PreviousLoanStatusId,
        s.LoanStatusId
from    ${beh.ods}.policyloanstatus s
join    ${beh.stg}.beh_policy_scv l on l.PolicyCode = s.PolicyCode
where   s.PreviousLoanStatusId is not null
  and   s.PreviousLoanStatusId <> s.LoanStatusId
  and   s.StatusFromDate >= to_date('${beh.cov_loan}')
  and   s.StatusFromDate <= to_date('${beh.last_me}')


## beh_fee_concession


In [ ]:
/****************************************************************************************************/
/*  Name        : stg.beh_fee_concession                                                            */
/*  Description : Motor renewals retained with the fee reduced from the offered fee (TYEURFees      */
/*                below RenEURFee on the renewal's own PolicyRenewal row), dated at the renewal     */
/*                sale date. Replaces the R2.E1 events, whose ads_v2 join to PolicyRenewal ignores  */
/*                the renewal date and flags every cycle of a policy.                               */
/*  Reads       : ods_v2.PolicyRenewal, stg.beh_policy_scv                                          */
/*  Key         : PolicyCode + LYPolicyRenewDateAdj                                                 */
/****************************************************************************************************/
create or replace table ${beh.stg}.beh_fee_concession as
select  l.SCV_Customer_Key,
        r.PolicyCode,
        r.LYPolicyRenewDateAdj,
        coalesce(r.TYReportingSaleDate, r.LYPolicyRenewDateAdj) as ActivityDate,
        r.RenEURFee - r.TYEURFees                               as ConcessionFeeEUR
from    ${beh.ods}.policyrenewal r
join    ${beh.stg}.beh_policy_scv l on l.PolicyCode = r.PolicyCode
where   r.LYPolicyTypeGroup = 'Motor'
  and   r.PolicyOfferNum = 1
  and   r.PolicyRetNum = 1
  and   r.TYEURFees < r.RenEURFee
  and   coalesce(r.TYReportingSaleDate, r.LYPolicyRenewDateAdj) <= to_date('${beh.last_me}')


## beh_sale


In [ ]:
/****************************************************************************************************/
/*  Name        : stg.beh_sale                                                                      */
/*  Description : New-business Motor, Home and Van sales, first sale per policy (rebooked MTAs and  */
/*                reversals out, as BEH_50), with IsWebAssist when the sale's channel is 'b) Web    */
/*                Assist' (curation #27: started online, completed with an agent, the A4.F1 /       */
/*                HA4.F1 rule). Linked by policy, so the GDPR quote wipe does not apply.            */
/*  Reads       : ods_v2.PolicySale, stg.beh_policy_scv                                             */
/*  Key         : PolicyCode                                                                        */
/****************************************************************************************************/
create or replace table ${beh.stg}.beh_sale as
select  l.SCV_Customer_Key,
        s.PolicyCode,
        min(s.ReportingSaleDate)                                                    as ActivityDate,
        min_by(case when s.Channel = 'b) Web Assist' then 1 else 0 end, s.ReportingSaleDate)  as IsWebAssist
from    ${beh.ods}.policysale s
join    ${beh.stg}.beh_policy_scv l on l.PolicyCode = s.PolicyCode
where   s.ReportingSaleType = 'New Business'
  and   s.PolicyTypeGroup in ('Motor', 'Home', 'Van')
  and   s.SaleCount > 0
  and   coalesce(s.MTARebookedFlag, 'N') <> 'Y'
  and   s.ReportingSaleDate <= to_date('${beh.last_me}')
group by l.SCV_Customer_Key, s.PolicyCode


## beh_doc_wf


In [ ]:
/****************************************************************************************************/
/*  Name        : stg.beh_doc_wf                                                                    */
/*  Description : MyChill document decisions (DocumentWorkflowStatus: one row per decision,         */
/*                accepted or rejected, with the upload attempts), dated on the decision, linked    */
/*                through the policy (curation #46).                                                */
/*  Reads       : ods_v2.DocumentWorkflowStatus, stg.beh_policy_scv                                 */
/*  Key         : Pk                                                                                */
/****************************************************************************************************/
create or replace table ${beh.stg}.beh_doc_wf as
select  w.Pk,
        l.SCV_Customer_Key,
        to_date(coalesce(w.EventTimestamp, w.AssignedDate))         as ActivityDate,
        case when w.IsRejected then 1 else 0 end                    as IsRejected,
        coalesce(w.Attempts, 0)                                     as Attempts
from    ${beh.ods}.documentworkflowstatus w
join    ${beh.stg}.beh_policy_scv l on l.PolicyCode = w.PolicyCode
where   (w.IsAccepted or w.IsRejected)
  and   to_date(coalesce(w.EventTimestamp, w.AssignedDate)) >= to_date('${beh.cov_docwf}')
  and   to_date(coalesce(w.EventTimestamp, w.AssignedDate)) <= to_date('${beh.last_me}')


## beh_doc_nonreceipt


In [ ]:
/****************************************************************************************************/
/*  Name        : stg.beh_doc_nonreceipt                                                            */
/*  Description : Documents not received (beh_codes.DOC_NONRECEIPT events) once per customer,       */
/*                policy and month, with the policy's gross premium at the previous month-end, for  */
/*                the premium-weighted reading (curation #53).                                      */
/*  Reads       : stg.beh_event, ods_v2.PolicyMonthEnd                                              */
/*  Key         : SCV_Customer_Key + PolicyCode + MonthNo                                           */
/****************************************************************************************************/
create or replace table ${beh.stg}.beh_doc_nonreceipt as
select  e.SCV_Customer_Key,
        e.EffectivePolicyCode                                           as PolicyCode,
        year(e.ActivityDate) * 12 + month(e.ActivityDate)               as MonthNo,
        min(e.ActivityDate)                                             as ActivityDate,
        max(coalesce(m.EURGrossPremClose, 0))                           as PremiumEUR
from    ${beh.stg}.beh_event e
left join ${beh.ods}.policymonthend m
        on  m.PolicyCode = e.EffectivePolicyCode
        and m.EffectiveDate = last_day(add_months(e.ActivityDate, -1))
where   e.doc_nonreceipt = 1
  and   e.EffectivePolicyCode is not null
group by e.SCV_Customer_Key, e.EffectivePolicyCode, year(e.ActivityDate) * 12 + month(e.ActivityDate)


## beh_consent


In [ ]:
/****************************************************************************************************/
/*  Name        : stg.beh_consent                                                                   */
/*  Description : Marketing consent statements (Mercury ConsentAction) linked to exactly one SCV    */
/*                customer through the email or phone key valid at the time. ActionId 1 = opt-in,   */
/*                2 = opt-out, 3 and 4 left out as the warehouse does (usp_Load_Fact_MercuryDNC).   */
/*                IsWithdrawal = an opt-out straight after an opt-in on the same contact and        */
/*                channel (curation #64). Only statements made through the contact-centre           */
/*                interfaces (AppName CCI EXO, CCI SE) count, because quote forms (MOTOR FQ) were   */
/*                logged only in 2022 and 2023 and would make withdrawals jump in those years. The  */
/*                earlier opt-in may come from any app.                                             */
/*  Reads       : ods_v2.ConsentAction, ads_v2.ContactScvLink                                       */
/*  Key         : ActionTrackId                                                                     */
/****************************************************************************************************/
create or replace table ${beh.stg}.beh_consent as
select  a.ActionTrackId,
        min(l.SCV_Customer_Key)                         as SCV_Customer_Key,
        to_date(a.ConsentRecordedDateTime)              as ActivityDate,
        max(a.IsWithdrawal)                             as IsWithdrawal
from    (select  ActionTrackId,
                 ConsentRecordedDateTime,
                 case when EmailKey is not null then 'EmailKey' else 'PhoneKey' end   as KeyType,
                 coalesce(EmailKey, PhoneKey)                                         as ContactKey,
                 AppName,
                 case when ActionId = 2
                       and lag(ActionId) over (partition by coalesce(EmailKey, PhoneKey), ChannelId
                                               order by ConsentRecordedDateTime, ActionTrackId) = 1
                      then 1 else 0 end                                               as IsWithdrawal
         from    ${beh.ods}.consentaction
         where   ActionId in (1, 2)
           and   coalesce(EmailKey, PhoneKey) is not null
           and   to_date(ConsentRecordedDateTime) <= to_date('${beh.last_me}')) a
join    ${beh.ads}.contactscvlink l
        on  l.ContactKeyType = a.KeyType
        and l.ContactKey = a.ContactKey
        and a.ConsentRecordedDateTime >= l.ValidFrom
        and (l.ValidTo is null or a.ConsentRecordedDateTime < l.ValidTo)
where   a.AppName in ('CCI EXO', 'CCI SE')
  and   to_date(a.ConsentRecordedDateTime) >= to_date('${beh.cov_consent}')
group by a.ActionTrackId, to_date(a.ConsentRecordedDateTime)
having  count(distinct l.SCV_Customer_Key) = 1


## beh_portal_month


In [ ]:
/****************************************************************************************************/
/*  Name        : stg.beh_portal_month                                                              */
/*  Description : Renewal portal log-ins per customer and month: the last failed and the last       */
/*                successful log-in time (RenewalPortalEvent LoginFailed / LoginSuccess, linked     */
/*                portfolio -> policy -> SCV, portfolios on one customer only). BEH_30 reads it     */
/*                for 'failed log-in with no later log-in' (curation #52).                          */
/*  Reads       : ods_v2.RenewalPortalEvent, ods_v2.ClientPortfolio, stg.beh_policy_scv             */
/*  Key         : SCV_Customer_Key + MonthNo                                                        */
/****************************************************************************************************/
create or replace table ${beh.stg}.beh_portal_month as
select  k.SCV_Customer_Key,
        year(e.EventDateTime) * 12 + month(e.EventDateTime)                         as MonthNo,
        max(case when e.EventType = 'LoginFailed' then e.EventDateTime end)         as LastFailTs,
        max(case when e.EventType = 'LoginSuccess' then e.EventDateTime end)        as LastSuccessTs
from    ${beh.ods}.renewalportalevent e
join    (select  c.ClientCode,
                 min(l.SCV_Customer_Key)     as SCV_Customer_Key
         from    ${beh.ods}.clientportfolio c
         join    ${beh.stg}.beh_policy_scv l on l.PolicyCode = c.PolicyCode
         group by c.ClientCode
         having  count(distinct l.SCV_Customer_Key) = 1) k on k.ClientCode = e.PortfolioCode
where   e.EventType in ('LoginFailed', 'LoginSuccess')
  and   e.EventDateTime >= to_timestamp('${beh.cov_portal}')
  and   e.EventDateTime < date_add(to_date('${beh.last_me}'), 1)
group by k.SCV_Customer_Key, year(e.EventDateTime) * 12 + month(e.EventDateTime)


## beh_mfq_seed


In [ ]:
/****************************************************************************************************/
/*  Name        : stg.beh_mfq_seed                                                                  */
/*  Description : MFQ quotes linked to a policyholder: ods.scveadmlinkfinal SourceSystemId 2,       */
/*                SubSourceSystemId 1 holds the proposer's driver id as                             */
/*                ChillSourceCustomerReference, which leads to the quote query and its guid         */
/*                (ods_v2.QuoteMotorQuery). Customers who hold a policy in the build only.          */
/*                KnownFrom is when the link could be known: before 2026-05-06 the GDPR wipe left   */
/*                a link only on quotes that became a sale, so it is the sale date (PolicySale on   */
/*                the quote guid); from 2026-05-06, or with no sale, the quote date.                */
/*  Reads       : ods.scveadmlinkfinal, ods_v2.QuoteMotorQuery, ods_v2.QuoteMotor,                  */
/*                ods_v2.PolicySale, beh.CustomerMonth                                              */
/*  Key         : SCV_Customer_Key + GuidKey                                                        */
/****************************************************************************************************/
create or replace table ${beh.stg}.beh_mfq_seed as
select  l.SCV_Customer_Key,
        q.GuidKey,
        min(case when to_date(m.FirstCreatedDateTimeUtc) >= date'2026-05-06' then to_date(m.FirstCreatedDateTimeUtc)
                 else coalesce(sd.SaleDate, to_date(m.FirstCreatedDateTimeUtc)) end)    as KnownFrom
from    ${beh.link}.scveadmlinkfinal l
join    ${beh.ods}.quotemotorquery q
        on  q.ProposerQuoteDriverId = try_cast(l.ChillSourceCustomerReference as bigint)
join    (select distinct SCV_Customer_Key from ${beh.target}.CustomerMonth) c
        on  c.SCV_Customer_Key = l.SCV_Customer_Key
join    ${beh.ods}.quotemotor m on m.GuidKey = q.GuidKey
left join (select lower(trim(QuoteQueryGuid)) as GuidKey, min(ReportingSaleDate) as SaleDate
           from   ${beh.ods}.policysale
           where  nullif(trim(QuoteQueryGuid), '') is not null
           group by lower(trim(QuoteQueryGuid))) sd on sd.GuidKey = q.GuidKey
where   l.SourceSystemId = 2
  and   l.SubSourceSystemId = 1
  and   q.GuidKey is not null
group by l.SCV_Customer_Key, q.GuidKey


## beh_mfq_retrieval


In [ ]:
/****************************************************************************************************/
/*  Name        : stg.beh_mfq_retrieval                                                             */
/*  Description : Each MFQ quote that retrieved an earlier quote, with the earlier quote's guid     */
/*                (RetrieveSessionToken).                                                           */
/*  Reads       : ods_v2.QuoteMotorQuery                                                            */
/*  Key         : GuidKey + RetrievedGuidKey                                                        */
/****************************************************************************************************/
create or replace table ${beh.stg}.beh_mfq_retrieval as
select  distinct GuidKey,
        RetrievedGuidKey
from    ${beh.ods}.quotemotorquery
where   RetrievedGuidKey is not null
  and   GuidKey is not null
  and   RetrievedGuidKey <> GuidKey


## beh_mfq_chain1


In [ ]:
/****************************************************************************************************/
/*  Name        : stg.beh_mfq_chain1                                                                */
/*  Description : Linked quotes plus one step along the retrieval chain in each direction (quotes   */
/*                they retrieved, and later quotes that retrieved them).                            */
/*  Reads       : stg.beh_mfq_seed, stg.beh_mfq_retrieval                                           */
/*  Key         : SCV_Customer_Key + GuidKey                                                        */
/****************************************************************************************************/
create or replace table ${beh.stg}.beh_mfq_chain1 as
select  SCV_Customer_Key, GuidKey, min(KnownFrom) as KnownFrom
from    (select  SCV_Customer_Key, GuidKey, KnownFrom from ${beh.stg}.beh_mfq_seed
         union all
         select  s.SCV_Customer_Key, r.GuidKey, s.KnownFrom
         from    ${beh.stg}.beh_mfq_seed s join ${beh.stg}.beh_mfq_retrieval r on r.RetrievedGuidKey = s.GuidKey
         union all
         select  s.SCV_Customer_Key, r.RetrievedGuidKey, s.KnownFrom
         from    ${beh.stg}.beh_mfq_seed s join ${beh.stg}.beh_mfq_retrieval r on r.GuidKey = s.GuidKey) u
group by SCV_Customer_Key, GuidKey


## beh_mfq_chain2


In [ ]:
/****************************************************************************************************/
/*  Name        : stg.beh_mfq_chain2                                                                */
/*  Description : A second step along the retrieval chain in each direction.                        */
/*  Reads       : stg.beh_mfq_chain1, stg.beh_mfq_retrieval                                         */
/*  Key         : SCV_Customer_Key + GuidKey                                                        */
/****************************************************************************************************/
create or replace table ${beh.stg}.beh_mfq_chain2 as
select  SCV_Customer_Key, GuidKey, min(KnownFrom) as KnownFrom
from    (select  SCV_Customer_Key, GuidKey, KnownFrom from ${beh.stg}.beh_mfq_chain1
         union all
         select  s.SCV_Customer_Key, r.GuidKey, s.KnownFrom
         from    ${beh.stg}.beh_mfq_chain1 s join ${beh.stg}.beh_mfq_retrieval r on r.RetrievedGuidKey = s.GuidKey
         union all
         select  s.SCV_Customer_Key, r.RetrievedGuidKey, s.KnownFrom
         from    ${beh.stg}.beh_mfq_chain1 s join ${beh.stg}.beh_mfq_retrieval r on r.GuidKey = s.GuidKey) u
group by SCV_Customer_Key, GuidKey


## beh_mfq_quote


In [ ]:
/****************************************************************************************************/
/*  Name        : stg.beh_mfq_quote                                                                 */
/*  Description : One row per MFQ quote linked to exactly one policyholder (quotes reached from     */
/*                two customers are left out), with the curation's readings, all known when the     */
/*                quote is made: whether it retrieved an earlier quote (IsRetrieval), whether that  */
/*                earlier quote was 7 days or more old (Returned), insurers priced, voluntary       */
/*                excess, the cover chosen at the selection steps (01 Comprehensive, 02 TPFT) and   */
/*                the first and last device. Attributes come from the quote's latest query row      */
/*                (QuoteMotor), never an earlier one. ActivityDate is the later of the quote date   */
/*                and KnownFrom, so a quote linked only through a later sale counts from the sale,  */
/*                never in a month-end before it (no look-ahead).                                   */
/*  Reads       : stg.beh_mfq_chain2, stg.beh_mfq_retrieval, ods_v2.QuoteMotor,                     */
/*                ods_v2.QuoteMotorStep, ods_v2.QuoteMotorSummary                                   */
/*  Key         : GuidKey                                                                           */
/****************************************************************************************************/
create or replace table ${beh.stg}.beh_mfq_quote as
select  g.SCV_Customer_Key,
        g.GuidKey,
        m.FirstCreatedDateTimeUtc                                                   as QuoteTs,
        g.KnownFrom,
        greatest(to_date(m.FirstCreatedDateTimeUtc), g.KnownFrom)                   as ActivityDate,
        case when rr.GuidKey is not null then 1 else 0 end                          as IsRetrieval,
        m.InsurerCount,
        case when m.FirstCreatedDateTimeUtc >= rr.EarliestRetrievedTs + interval 7 days then 1 else 0 end
                                                                                    as Returned,
        m.CoverVoluntaryExcess                                                      as Excess,
        c.ChosenCover,
        case when s.FirstDevice is not null and s.LastDevice is not null then 1 else 0 end as DeviceTagged,
        case when s.FirstDevice is not null and s.LastDevice is not null and s.FirstDevice <> s.LastDevice
             then 1 else 0 end                                                      as DeviceSwitch
from    (select GuidKey, min(SCV_Customer_Key) as SCV_Customer_Key, min(KnownFrom) as KnownFrom
         from   ${beh.stg}.beh_mfq_chain2
         group by GuidKey
         having count(distinct SCV_Customer_Key) = 1) g
join    ${beh.ods}.quotemotor m on m.GuidKey = g.GuidKey
left join (select GuidKey, max_by(CoverCode, StepDateTimeUtc) as ChosenCover
           from   ${beh.ods}.quotemotorstep
           where  CoverCode in ('01', '02')
           group by GuidKey) c on c.GuidKey = g.GuidKey
left join ${beh.ods}.quotemotorsummary s on s.GuidKey = g.GuidKey
left join (select r.GuidKey, min(m2.FirstCreatedDateTimeUtc) as EarliestRetrievedTs
           from   ${beh.stg}.beh_mfq_retrieval r
           join   ${beh.ods}.quotemotor m2 on m2.GuidKey = r.RetrievedGuidKey
           group by r.GuidKey) rr on rr.GuidKey = g.GuidKey
where   to_date(m.FirstCreatedDateTimeUtc) >= to_date('${beh.cov_mfq}')
  and   greatest(to_date(m.FirstCreatedDateTimeUtc), g.KnownFrom) <= to_date('${beh.last_me}')


## beh_mfq_quote_x


In [ ]:
/****************************************************************************************************/
/*  Name        : stg.beh_mfq_quote_x                                                               */
/*  Description : Linked MFQ quotes with the cover change against the customer's previous quote     */
/*                that has a chosen cover (curation #11: compared across quotes, never within       */
/*                one).                                                                             */
/*  Reads       : stg.beh_mfq_quote                                                                 */
/*  Key         : GuidKey                                                                           */
/****************************************************************************************************/
create or replace table ${beh.stg}.beh_mfq_quote_x as
select  q.*,
        case when q.ChosenCover is not null and p.PrevCover is not null then 1 else 0 end      as CoverCompared,
        case when p.PrevCover = '01' and q.ChosenCover = '02' then 1 else 0 end                as CoverDown,
        case when p.PrevCover = '02' and q.ChosenCover = '01' then 1 else 0 end                as CoverUp
from    ${beh.stg}.beh_mfq_quote q
left join (select GuidKey,
                  lag(ChosenCover) over (partition by SCV_Customer_Key order by ActivityDate, QuoteTs, GuidKey) as PrevCover
           from   ${beh.stg}.beh_mfq_quote
           where  ChosenCover is not null) p on p.GuidKey = q.GuidKey


## beh_web_cookie


In [ ]:
/****************************************************************************************************/
/*  Name        : stg.beh_web_cookie                                                                */
/*  Description : GA4 cookies (ClientID) tied to exactly one customer through an MFQ quote linked   */
/*                to that customer (stg.beh_mfq_quote: their own quotes from May 2026, and before   */
/*                that the quotes on the way to an online purchase), known from the date the        */
/*                quote's link became known. Curation #54, #57, #74.                                */
/*  Reads       : ods_v2.WebQuoteClick, stg.beh_mfq_quote                                           */
/*  Key         : ClientID                                                                          */
/****************************************************************************************************/
create or replace table ${beh.stg}.beh_web_cookie as
select  w.ClientID,
        min(q.SCV_Customer_Key)             as SCV_Customer_Key,
        min(q.ActivityDate)                 as KnownFrom
from    ${beh.ods}.webquoteclick w
join    ${beh.stg}.beh_mfq_quote q on q.GuidKey = w.GuidKey
where   nullif(trim(w.ClientID), '') is not null
group by w.ClientID
having  count(distinct q.SCV_Customer_Key) = 1


## beh_web_session_email


In [ ]:
/****************************************************************************************************/
/*  Name        : stg.beh_web_session_email                                                         */
/*  Description : GA4 sessions whose cookie the warehouse ties to an email from its MFQ quotes      */
/*                (hashed in ods_v2), linked to the one customer holding that email at the session  */
/*                start (ads_v2.ContactScvLink). Sessions whose email several customers hold at     */
/*                that time are left out.                                                           */
/*  Reads       : ods_v2.WebSession, ads_v2.ContactScvLink (EmailKey)                               */
/*  Key         : ClientID + SessionID                                                              */
/****************************************************************************************************/
create or replace table ${beh.stg}.beh_web_session_email as
select  s.ClientID,
        s.SessionID,
        min(l.SCV_Customer_Key)             as SCV_Customer_Key
from    ${beh.ods}.websession s
join    ${beh.ads}.contactscvlink l
        on  l.ContactKeyType = 'EmailKey'
        and l.ContactKey = s.EmailKey
        and s.SessionStartUtc >= l.ValidFrom
        and (l.ValidTo is null or s.SessionStartUtc < l.ValidTo)
where   s.EmailKey is not null
  and   s.LinkedContactCount = 1
  and   s.SessionDate >= to_date('${beh.cov_web}')
  and   s.SessionDate <= to_date('${beh.last_me}')
group by s.ClientID, s.SessionID
having  count(distinct l.SCV_Customer_Key) = 1


## beh_web_session


In [ ]:
/****************************************************************************************************/
/*  Name        : stg.beh_web_session                                                               */
/*  Description : Web sessions linked to a customer: by the cookie (a linked MFQ quote, from the    */
/*                date the link was known), else by the warehouse's email link, kept only while     */
/*                the person is a customer (holds a policy at the month-end of the session):        */
/*                sessions before a purchase are acquisition shopping, and 62% of email-linked      */
/*                sessions in 2025 were on keys holding nothing at the time. OtherProduct = 1 when  */
/*                the session viewed pages of a product the customer did not hold at the month-end  */
/*                of the session (Home, Van or Motor not held; Travel, Life, Gadget and Commercial  */
/*                are not tracked as holdings, so they always count). Curation #54, #57, #74.       */
/*  Reads       : ods_v2.WebSession, stg.beh_web_cookie, stg.beh_web_session_email,                 */
/*                beh.CustomerMonth                                                                 */
/*  Key         : ClientID + SessionID                                                              */
/****************************************************************************************************/
create or replace table ${beh.stg}.beh_web_session as
select  s.ClientID,
        s.SessionID,
        coalesce(c.SCV_Customer_Key, e.SCV_Customer_Key)                    as SCV_Customer_Key,
        case when c.SCV_Customer_Key is not null then 'cookie' else 'email' end as LinkedBy,
        s.SessionDate                                                        as ActivityDate,
        s.PageViews,
        s.ContentViews,
        s.HasQuoteReference,
        case when (s.ViewedHome = 1 and coalesce(h.HoldsHome, 0) = 0)
               or (s.ViewedVan = 1 and coalesce(h.HoldsVan, 0) = 0)
               or (s.ViewedMotor = 1 and coalesce(h.HoldsMotor, 0) = 0)
               or s.ViewedTravel = 1 or s.ViewedLife = 1 or s.ViewedGadget = 1 or s.ViewedCommercial = 1
             then 1 else 0 end                                               as OtherProduct
from    ${beh.ods}.websession s
left join ${beh.stg}.beh_web_cookie c
        on  c.ClientID = s.ClientID
        and s.SessionDate >= c.KnownFrom
left join ${beh.stg}.beh_web_session_email e
        on  e.ClientID = s.ClientID
        and e.SessionID = s.SessionID
join    ${beh.target}.customermonth h
        on  h.SCV_Customer_Key = coalesce(c.SCV_Customer_Key, e.SCV_Customer_Key)
        and h.MonthEndDate = last_day(s.SessionDate)
where   coalesce(c.SCV_Customer_Key, e.SCV_Customer_Key) is not null
  and   s.SessionDate >= to_date('${beh.cov_web}')
  and   s.SessionDate <= to_date('${beh.last_me}')


## beh_client_scv


In [ ]:
/****************************************************************************************************/
/*  Name        : stg.beh_client_scv                                                                */
/*  Description : Relay client code to SCV customer key, where the code links to exactly one        */
/*                customer.                                                                         */
/*  Reads       : ods.scveadmlinkfinal (SourceSystemId 1)                                           */
/*  Key         : ClientCode                                                                        */
/****************************************************************************************************/
create or replace table ${beh.stg}.beh_client_scv as
select  upper(trim(ChillSourceCustomerReference))  as ClientCode,
        min(SCV_Customer_Key)                       as SCV_Customer_Key
from    ${beh.link}.scveadmlinkfinal
where   SourceSystemId = 1
  and   SCV_Customer_Key is not null
  and   nullif(trim(ChillSourceCustomerReference), '') is not null
group by upper(trim(ChillSourceCustomerReference))
having  count(distinct SCV_Customer_Key) = 1


## beh_tx_day


In [ ]:
/****************************************************************************************************/
/*  Name        : stg.beh_tx_day                                                                    */
/*  Description : Days on which a customer had a new-business sale, a renewal or an MTA posting     */
/*                (for the 7 day rule of curation #56).                                             */
/*  Reads       : ods_v2.PolicySale, ods_v2.PolicyMtaDay, stg.beh_policy_scv                        */
/*  Key         : SCV_Customer_Key + TxDate                                                         */
/****************************************************************************************************/
create or replace table ${beh.stg}.beh_tx_day as
select  distinct l.SCV_Customer_Key,
        t.TxDate
from    (select PolicyCode, ReportingSaleDate as TxDate
         from   ${beh.ods}.policysale
         where  ReportingSaleType in ('New Business', 'Renewals')
         union all
         select PolicyCode, WorkDate
         from   ${beh.ods}.policymtaday
         where  MtaPostings <> 0) t
join    ${beh.stg}.beh_policy_scv l on l.PolicyCode = t.PolicyCode
where   t.TxDate >= date_sub(to_date('${beh.cov_tcall}'), 1)


## beh_tcall_base


In [ ]:
/****************************************************************************************************/
/*  Name        : stg.beh_tcall_base                                                                */
/*  Description : Inbound calls to the Motor renewal base's other-product and general service       */
/*                queues (beh_codes.MOTOR_BASE_OTHER_QUEUES, curation #56), tied to the customer    */
/*                by the client code on the call, else by the caller's number                       */
/*                (ads_v2.ContactScvLink, one customer at the call start). The client code is the   */
/*                record the agent opened: where both link and disagree (about 8% of calls) it is   */
/*                usually one person on two SCV keys or a household member.                         */
/*  Reads       : ods_v2.TelephonyCall, stg.beh_client_scv, ads_v2.ContactScvLink                   */
/*  Key         : CallIDKey                                                                         */
/****************************************************************************************************/
create or replace table ${beh.stg}.beh_tcall_base as
select  c.CallIDKey,
        coalesce(k.SCV_Customer_Key, p.SCV_Customer_Key)                    as SCV_Customer_Key,
        case when k.SCV_Customer_Key is not null then 'client code' else 'phone' end as LinkedBy,
        c.WorkGroup,
        to_date(c.CallStart)                                                 as CallDate
from    (select  CallIDKey,
                 min(CallStartDT)                                       as CallStart,
                 max(WorkGroup)                                         as WorkGroup,
                 max(PhoneKey)                                          as PhoneKey,
                 max(upper(nullif(trim(ClientCode), '')))               as ClientCode
         from    ${beh.ods}.telephonycall
         where   lower(CallDirection) = 'inbound'
           and   WorkGroup in ('INBOUND_SALES_HOME', 'INBOUND_SALES_VAN', 'INBOUND_SALES_LIFE', 'INBOUND_NON-STANDARD_MOTOR', 'INBOUND_Other', 'INBOUND_CS_Other', 'Chat - Customer Service')
           and   CallStartDT >= to_timestamp('${beh.cov_tcall}')
         group by CallIDKey) c
left join ${beh.stg}.beh_client_scv k on k.ClientCode = c.ClientCode
left join (select  c2.CallIDKey, min(l.SCV_Customer_Key) as SCV_Customer_Key
           from    (select CallIDKey, min(CallStartDT) as CallStart, max(PhoneKey) as PhoneKey
                    from   ${beh.ods}.telephonycall
                    where  lower(CallDirection) = 'inbound'
                      and  WorkGroup in ('INBOUND_SALES_HOME', 'INBOUND_SALES_VAN', 'INBOUND_SALES_LIFE', 'INBOUND_NON-STANDARD_MOTOR', 'INBOUND_Other', 'INBOUND_CS_Other', 'Chat - Customer Service')
                      and  CallStartDT >= to_timestamp('${beh.cov_tcall}')
                    group by CallIDKey) c2
           join    ${beh.ads}.contactscvlink l
                   on  l.ContactKeyType = 'PhoneKey'
                   and l.ContactKey = c2.PhoneKey
                   and c2.CallStart >= l.ValidFrom
                   and (l.ValidTo is null or c2.CallStart < l.ValidTo)
           group by c2.CallIDKey
           having  count(distinct l.SCV_Customer_Key) = 1) p on p.CallIDKey = c.CallIDKey
where   coalesce(k.SCV_Customer_Key, p.SCV_Customer_Key) is not null


## beh_tcall


In [ ]:
/****************************************************************************************************/
/*  Name        : stg.beh_tcall                                                                     */
/*  Description : The calls of stg.beh_tcall_base with the 7 day rule (curation #56): NoTx7 = 1     */
/*                when the customer had no sale, renewal or MTA from the call date to 7 days after  */
/*                it. ActivityDate is the call date plus 7, when that is known, so a call never     */
/*                counts at a month-end before its 7 days have passed.                              */
/*  Reads       : stg.beh_tcall_base, stg.beh_tx_day                                                */
/*  Key         : CallIDKey                                                                         */
/****************************************************************************************************/
create or replace table ${beh.stg}.beh_tcall as
select  b.CallIDKey,
        b.SCV_Customer_Key,
        b.LinkedBy,
        b.WorkGroup,
        b.CallDate,
        date_add(b.CallDate, 7)                                     as ActivityDate,
        case when max(x.TxDate) is null then 1 else 0 end           as NoTx7
from    ${beh.stg}.beh_tcall_base b
left join ${beh.stg}.beh_tx_day x
        on  x.SCV_Customer_Key = b.SCV_Customer_Key
        and x.TxDate between b.CallDate and date_add(b.CallDate, 7)
where   date_add(b.CallDate, 7) <= to_date('${beh.last_me}')
group by b.CallIDKey, b.SCV_Customer_Key, b.LinkedBy, b.WorkGroup, b.CallDate


## beh_friction_res


In [ ]:
/****************************************************************************************************/
/*  Name        : stg.beh_friction_res                                                              */
/*  Description : Friction events and their resolution (curation #49): one row per friction event   */
/*                on its date (IsEvent = 1), and one row on the date of the first later main-       */
/*                journey (spine) step with a real time in the same episode (IsResolved = 1), so a  */
/*                resolution counts only once it has happened. Only friction events with a real     */
/*                time can be resolved: a derived time (R4.F1 at due minus 60) cannot be ordered,   */
/*                and resolving it against renewal steps let the renewal outcome into readings      */
/*                before it (leak scan, run 24). Every friction code counts, Chill process issues   */
/*                included; Family groups the codes (beh_codes.FRICTION_FAMILY).                    */
/*  Reads       : stg.beh_event                                                                     */
/****************************************************************************************************/
create or replace table ${beh.stg}.beh_friction_res as
select  f.SCV_Customer_Key, f.ActivityDate, f.Family, f.InLife, 1 as IsEvent, 0 as IsResolved
from    (select e.SCV_Customer_Key, e.ActivityDate, e.InLife, case when e.EventTypeId in ('A5.F1', 'R4.F1', 'HA5.F1', 'M4.F1', 'HR4.F1', 'HM4.F1', 'VR4.F1', 'VM4.F1') then 'payment' when e.EventTypeId in ('R2.F1') then 'login' when e.EventTypeId in ('M1.F1', 'R3b.F1', 'D1.F1', 'A0.F1', 'C1a.F1', 'HR3b.F1', 'HD1.F1', 'HM1.F1', 'HA0.F1', 'VM1.F1', 'VR3b.F1', 'VD1.F1', 'VA0.F1', 'D3.F1', 'HD3.F1', 'VD3.F1', 'CL3.F1', 'R3.B1.1.F1', 'HR3.B1.1.F1', 'VR3.B1.1.F1', 'HR3.B1.2.F1', 'VR3.B1.2.F1', 'R3.B1.2.F1') then 'contact' when e.EventTypeId in ('R6.F1', 'HR6.F1', 'A6.F1', 'A6.B1.F1', 'A7.F1', 'R4.B2.F1', 'HA6.F1', 'HR4.B2.F1', 'HM4.B1.F1', 'VA6.B1.F1', 'VM3.B1.F1', 'VR4.B2.F1') then 'documents' when e.EventTypeId in ('A2.F1', 'HA2.F1', 'A3.F1', 'HA3.F1', 'HA1.F1', 'A4.F1', 'HA4.F1', 'R2.B1.F1', 'HR2.B1.F1', 'R2.B1.3.F1', 'HR2.B1.2.F1', 'R2.B1.4.F1', 'HR2.B1.3.F1', 'HR2.F1') then 'quote' when e.EventTypeId in ('R0.F1', 'R0.F2', 'HR0.F1', 'HR0.F2', 'HR0.F3', 'R1.E1.F3', 'HR1.E1.F1') then 'process' else 'other' end as Family
         from   ${beh.stg}.beh_event e
         where  e.StepType = 'friction' and e.EpisodeKey is not null
           and  e.ActivityDate <= to_date('${beh.last_me}')) f
union all
select  r.SCV_Customer_Key, to_date(r.ResolvedTs), r.Family, r.InLife, 0, 1
from    (select  e.EventId, e.SCV_Customer_Key, e.InLife, case when e.EventTypeId in ('A5.F1', 'R4.F1', 'HA5.F1', 'M4.F1', 'HR4.F1', 'HM4.F1', 'VR4.F1', 'VM4.F1') then 'payment' when e.EventTypeId in ('R2.F1') then 'login' when e.EventTypeId in ('M1.F1', 'R3b.F1', 'D1.F1', 'A0.F1', 'C1a.F1', 'HR3b.F1', 'HD1.F1', 'HM1.F1', 'HA0.F1', 'VM1.F1', 'VR3b.F1', 'VD1.F1', 'VA0.F1', 'D3.F1', 'HD3.F1', 'VD3.F1', 'CL3.F1', 'R3.B1.1.F1', 'HR3.B1.1.F1', 'VR3.B1.1.F1', 'HR3.B1.2.F1', 'VR3.B1.2.F1', 'R3.B1.2.F1') then 'contact' when e.EventTypeId in ('R6.F1', 'HR6.F1', 'A6.F1', 'A6.B1.F1', 'A7.F1', 'R4.B2.F1', 'HA6.F1', 'HR4.B2.F1', 'HM4.B1.F1', 'VA6.B1.F1', 'VM3.B1.F1', 'VR4.B2.F1') then 'documents' when e.EventTypeId in ('A2.F1', 'HA2.F1', 'A3.F1', 'HA3.F1', 'HA1.F1', 'A4.F1', 'HA4.F1', 'R2.B1.F1', 'HR2.B1.F1', 'R2.B1.3.F1', 'HR2.B1.2.F1', 'R2.B1.4.F1', 'HR2.B1.3.F1', 'HR2.F1') then 'quote' when e.EventTypeId in ('R0.F1', 'R0.F2', 'HR0.F1', 'HR0.F2', 'HR0.F3', 'R1.E1.F3', 'HR1.E1.F1') then 'process' else 'other' end as Family,
                 min(x.EventDateTime) as ResolvedTs
         from    ${beh.stg}.beh_event e
         join    ${beh.stg}.beh_event x
                 on  x.EpisodeKey = e.EpisodeKey
                 and x.StepType = 'spine'
                 and x.TimeUsable = 1
                 and x.EventDateTime > e.EventDateTime
         where   e.StepType = 'friction' and e.EpisodeKey is not null and e.TimeUsable = 1
         group by e.EventId, e.SCV_Customer_Key, e.InLife, case when e.EventTypeId in ('A5.F1', 'R4.F1', 'HA5.F1', 'M4.F1', 'HR4.F1', 'HM4.F1', 'VR4.F1', 'VM4.F1') then 'payment' when e.EventTypeId in ('R2.F1') then 'login' when e.EventTypeId in ('M1.F1', 'R3b.F1', 'D1.F1', 'A0.F1', 'C1a.F1', 'HR3b.F1', 'HD1.F1', 'HM1.F1', 'HA0.F1', 'VM1.F1', 'VR3b.F1', 'VD1.F1', 'VA0.F1', 'D3.F1', 'HD3.F1', 'VD3.F1', 'CL3.F1', 'R3.B1.1.F1', 'HR3.B1.1.F1', 'VR3.B1.1.F1', 'HR3.B1.2.F1', 'VR3.B1.2.F1', 'R3.B1.2.F1') then 'contact' when e.EventTypeId in ('R6.F1', 'HR6.F1', 'A6.F1', 'A6.B1.F1', 'A7.F1', 'R4.B2.F1', 'HA6.F1', 'HR4.B2.F1', 'HM4.B1.F1', 'VA6.B1.F1', 'VM3.B1.F1', 'VR4.B2.F1') then 'documents' when e.EventTypeId in ('A2.F1', 'HA2.F1', 'A3.F1', 'HA3.F1', 'HA1.F1', 'A4.F1', 'HA4.F1', 'R2.B1.F1', 'HR2.B1.F1', 'R2.B1.3.F1', 'HR2.B1.2.F1', 'R2.B1.4.F1', 'HR2.B1.3.F1', 'HR2.F1') then 'quote' when e.EventTypeId in ('R0.F1', 'R0.F2', 'HR0.F1', 'HR0.F2', 'HR0.F3', 'R1.E1.F3', 'HR1.E1.F1') then 'process' else 'other' end) r
where   to_date(r.ResolvedTs) <= to_date('${beh.last_me}')


## beh_offer_msg


In [ ]:
/****************************************************************************************************/
/*  Name        : stg.beh_offer_msg                                                                 */
/*  Description : Offer campaigns sent to the customer (build/offer_campaigns.py: promotions and    */
/*                offers, renewal price offers and winbacks Chill confirmed on 2 Nov 2026), with    */
/*                the send date (the click date, else the first of the month, where the send date   */
/*                is missing) and the click date. Curation #61.                                     */
/*  Reads       : ads_v2.CampaignHistory                                                            */
/*  Key         : CH_Id                                                                             */
/****************************************************************************************************/
create or replace table ${beh.stg}.beh_offer_msg as
select  h.CH_Id,
        h.SCV_Customer_Key,
        o.OfferClass,
        coalesce(to_date(h.min_SentDateTime), to_date(h.min_ClickDateTime),
                 to_date(concat(cast(h.MonthId as string), '01'), 'yyyyMMdd'))      as SendDate,
        to_date(h.min_ClickDateTime)                                               as ClickDate
from    ${beh.ads}.campaignhistory h
join    (select * from values
        (18808837, '', 'Winback'),
        (18808837, 'EMAIL', 'Winback'),
        (18809049, '', 'Winback'),
        (18809049, 'EMAIL', 'Winback'),
        (19092218, '', 'Winback'),
        (19092218, 'EMAIL', 'Winback'),
        (19092295, '', 'Winback'),
        (19092295, 'EMAIL', 'Winback'),
        (44597174, '', 'Promotion / offer'),
        (44597174, 'EMAIL', 'Promotion / offer'),
        (48606613, '', 'Winback'),
        (48606613, 'EMAIL', 'Winback'),
        (48606779, '', 'Winback'),
        (48606779, 'EMAIL', 'Winback'),
        (63711599, 'EMAIL', 'Winback'),
        (63711628, 'EMAIL', 'Winback'),
        (63711639, 'EMAIL', 'Winback'),
        (63711659, 'SMS', 'Winback'),
        (63711687, 'SMS', 'Winback'),
        (63711691, 'SMS', 'Winback'),
        (67374558, 'EMAIL', 'Winback'),
        (67548801, 'SMS', 'Promotion / offer'),
        (68105519, 'SMS', 'Promotion / offer'),
        (68132673, 'EMAIL', 'Winback'),
        (68452874, 'EMAIL', 'Winback'),
        (68475995, 'SMS', 'Winback'),
        (68501704, 'SMS', 'Winback'),
        (68501766, 'SMS', 'Winback'),
        (68525411, 'SMS', 'Winback'),
        (68525444, 'SMS', 'Winback'),
        (68530173, 'EMAIL', 'Winback'),
        (68851506, 'EMAIL', 'Promotion / offer'),
        (68851506, 'SMS', 'Promotion / offer'),
        (71925678, 'SMS', 'Winback'),
        (87067751, 'EMAIL', 'Promotion / offer'),
        (87067751, 'SMS', 'Promotion / offer'),
        (98797327, 'EMAIL', 'Winback'),
        (98798208, 'EMAIL', 'Winback'),
        (98800485, 'EMAIL', 'Winback'),
        (107055676, 'EMAIL', 'Promotion / offer'),
        (107707073, 'EMAIL', 'Promotion / offer'),
        (109163704, 'EMAIL', 'Promotion / offer'),
        (110812550, 'EMAIL', 'Winback'),
        (110814750, 'EMAIL', 'Winback'),
        (110814750, 'SMS', 'Winback'),
        (110819054, 'EMAIL', 'Winback'),
        (110819054, 'SMS', 'Winback'),
        (110823922, 'EMAIL', 'Winback'),
        (110823922, 'SMS', 'Winback'),
        (112159385, 'EMAIL', 'Winback'),
        (112159385, 'SMS', 'Winback'),
        (113479513, 'EMAIL', 'Promotion / offer'),
        (113479613, 'EMAIL', 'Promotion / offer'),
        (115885500, 'EMAIL', 'Winback'),
        (115885500, 'SMS', 'Winback'),
        (116063331, 'EMAIL', 'Winback'),
        (116063331, 'SMS', 'Winback'),
        (116930789, 'EMAIL', 'Winback'),
        (116930789, 'SMS', 'Winback'),
        (117175243, 'EMAIL', 'Promotion / offer'),
        (117190330, 'EMAIL', 'Promotion / offer'),
        (120278224, 'EMAIL', 'Promotion / offer'),
        (120278241, 'EMAIL', 'Promotion / offer'),
        (120445640, 'EMAIL', 'Promotion / offer'),
        (120939981, 'EMAIL', 'Promotion / offer'),
        (121141891, 'EMAIL', 'Promotion / offer'),
        (121255420, 'EMAIL', 'Promotion / offer'),
        (121865478, 'EMAIL', 'Promotion / offer'),
        (123534957, 'EMAIL', 'Promotion / offer'),
        (123874989, 'EMAIL', 'Promotion / offer'),
        (124098184, 'EMAIL', 'Promotion / offer'),
        (124331028, 'EMAIL', 'Promotion / offer'),
        (124549934, 'EMAIL', 'Promotion / offer'),
        (124833278, 'EMAIL', 'Promotion / offer'),
        (125030052, 'EMAIL', 'Promotion / offer'),
        (125347524, 'EMAIL', 'Promotion / offer'),
        (125609125, 'EMAIL', 'Promotion / offer'),
        (125666835, 'EMAIL', 'Promotion / offer'),
        (125710045, 'EMAIL', 'Promotion / offer'),
        (125781915, 'EMAIL', 'Promotion / offer'),
        (125930757, 'EMAIL', 'Promotion / offer'),
        (126100389, 'EMAIL', 'Promotion / offer'),
        (126215466, 'EMAIL', 'Renewal price offer'),
        (126217041, 'SMS', 'Renewal price offer'),
        (126217213, 'SMS', 'Renewal price offer'),
        (126217242, 'SMS', 'Renewal price offer'),
        (126307399, 'EMAIL', 'Promotion / offer'),
        (126334986, 'EMAIL', 'Renewal price offer'),
        (126335150, 'SMS', 'Renewal price offer'),
        (126447488, 'EMAIL', 'Renewal price offer'),
        (126447504, 'SMS', 'Renewal price offer'),
        (126522844, 'EMAIL', 'Promotion / offer'),
        (126526851, 'EMAIL', 'Renewal price offer'),
        (126647337, 'EMAIL', 'Renewal price offer'),
        (126649694, 'SMS', 'Renewal price offer'),
        (126725525, 'SMS', 'Renewal price offer'),
        (126725901, 'EMAIL', 'Renewal price offer'),
        (126746855, 'EMAIL', 'Renewal price offer'),
        (126748450, 'SMS', 'Renewal price offer'),
        (126808313, 'EMAIL', 'Renewal price offer'),
        (126808322, 'EMAIL', 'Renewal price offer'),
        (126808551, 'SMS', 'Renewal price offer'),
        (126808595, 'SMS', 'Renewal price offer'),
        (126856858, 'EMAIL', 'Promotion / offer'),
        (126857712, 'SMS', 'Renewal price offer'),
        (126857747, 'EMAIL', 'Renewal price offer'),
        (126857804, 'EMAIL', 'Renewal price offer'),
        (126867734, 'EMAIL', 'Renewal price offer'),
        (126867753, 'EMAIL', 'Renewal price offer'),
        (126868427, 'EMAIL', 'Renewal price offer'),
        (126868519, 'SMS', 'Renewal price offer'),
        (126868609, 'SMS', 'Renewal price offer'),
        (126868612, 'SMS', 'Renewal price offer'),
        (126898178, 'EMAIL', 'Renewal price offer'),
        (126898181, 'EMAIL', 'Renewal price offer'),
        (126898220, 'SMS', 'Renewal price offer'),
        (126898249, 'EMAIL', 'Renewal price offer'),
        (126898252, 'SMS', 'Renewal price offer'),
        (126898292, 'SMS', 'Renewal price offer'),
        (126945779, 'EMAIL', 'Renewal price offer'),
        (126945964, 'EMAIL', 'Renewal price offer'),
        (126945973, 'SMS', 'Renewal price offer'),
        (126945981, 'SMS', 'Renewal price offer'),
        (126965481, 'EMAIL', 'Promotion / offer'),
        (127011006, 'EMAIL', 'Renewal price offer'),
        (127011236, 'SMS', 'Renewal price offer'),
        (127011334, 'EMAIL', 'Renewal price offer'),
        (127011406, 'SMS', 'Renewal price offer'),
        (127061883, 'EMAIL', 'Renewal price offer'),
        (127061916, 'SMS', 'Renewal price offer'),
        (127166886, 'EMAIL', 'Renewal price offer'),
        (127166906, 'SMS', 'Renewal price offer'),
        (127185666, 'EMAIL', 'Renewal price offer'),
        (127288244, 'SMS', 'Renewal price offer'),
        (127288439, 'EMAIL', 'Renewal price offer'),
        (127309249, 'EMAIL', 'Renewal price offer'),
        (127309346, 'SMS', 'Renewal price offer'),
        (127371296, 'EMAIL', 'Renewal price offer'),
        (127373355, 'SMS', 'Renewal price offer'),
        (127409787, 'EMAIL', 'Renewal price offer'),
        (127425904, 'EMAIL', 'Renewal price offer'),
        (127507446, 'EMAIL', 'Renewal price offer'),
        (127509501, 'EMAIL', 'Renewal price offer'),
        (127525271, 'EMAIL', 'Renewal price offer'),
        (127525474, 'SMS', 'Renewal price offer'),
        (127558874, 'EMAIL', 'Renewal price offer'),
        (127558959, 'SMS', 'Renewal price offer'),
        (127609591, 'EMAIL', 'Renewal price offer'),
        (127629786, 'EMAIL', 'Renewal price offer'),
        (127630087, 'EMAIL', 'Renewal price offer'),
        (127650480, 'EMAIL', 'Renewal price offer'),
        (127650516, 'SMS', 'Renewal price offer'),
        (127669153, 'EMAIL', 'Renewal price offer'),
        (127737577, 'EMAIL', 'Renewal price offer'),
        (127755575, 'EMAIL', 'Renewal price offer'),
        (127755728, 'SMS', 'Renewal price offer'),
        (127766630, 'SMS', 'Renewal price offer'),
        (127766634, 'EMAIL', 'Renewal price offer'),
        (127806186, 'EMAIL', 'Renewal price offer'),
        (127809543, 'EMAIL', 'Promotion / offer'),
        (127850180, 'EMAIL', 'Renewal price offer'),
        (127867200, 'EMAIL', 'Renewal price offer'),
        (127883411, 'EMAIL', 'Renewal price offer'),
        (127884013, 'SMS', 'Renewal price offer'),
        (127902495, 'EMAIL', 'Renewal price offer'),
        (127921304, 'SMS', 'Renewal price offer'),
        (127955986, 'EMAIL', 'Promotion / offer'),
        (128008501, 'EMAIL', 'Renewal price offer'),
        (128008554, 'EMAIL', 'Renewal price offer'),
        (128041290, 'EMAIL', 'Renewal price offer'),
        (128041481, 'SMS', 'Renewal price offer'),
        (128096168, 'EMAIL', 'Renewal price offer'),
        (128115104, 'EMAIL', 'Renewal price offer'),
        (128137398, 'EMAIL', 'Renewal price offer'),
        (128154197, 'EMAIL', 'Renewal price offer'),
        (128222309, 'EMAIL', 'Renewal price offer'),
        (128242043, 'EMAIL', 'Renewal price offer'),
        (128259167, 'EMAIL', 'Renewal price offer'),
        (128259401, 'SMS', 'Renewal price offer'),
        (128279639, 'EMAIL', 'Renewal price offer'),
        (128367072, 'EMAIL', 'Renewal price offer'),
        (128367304, 'EMAIL', 'Renewal price offer'),
        (128386723, 'EMAIL', 'Promotion / offer'),
        (128386733, 'EMAIL', 'Promotion / offer'),
        (128387157, 'EMAIL', 'Renewal price offer'),
        (128387286, 'SMS', 'Renewal price offer'),
        (128402933, 'EMAIL', 'Renewal price offer'),
        (128406145, 'SMS', 'Renewal price offer'),
        (128480852, 'EMAIL', 'Renewal price offer'),
        (128514016, 'EMAIL', 'Renewal price offer'),
        (128514662, 'EMAIL', 'Renewal price offer'),
        (128550279, 'EMAIL', 'Renewal price offer'),
        (128552674, 'EMAIL', 'Promotion / offer'),
        (128633921, 'EMAIL', 'Renewal price offer'),
        (128653076, 'EMAIL', 'Renewal price offer'),
        (128653160, 'SMS', 'Renewal price offer'),
        (128715390, 'EMAIL', 'Renewal price offer'),
        (128732265, 'EMAIL', 'Renewal price offer'),
        (128749327, 'EMAIL', 'Renewal price offer'),
        (128749432, 'SMS', 'Renewal price offer'),
        (128765122, 'SMS', 'Renewal price offer'),
        (128765462, 'EMAIL', 'Renewal price offer'),
        (128828334, 'EMAIL', 'Renewal price offer'),
        (128828773, 'EMAIL', 'Renewal price offer'),
        (128843762, 'EMAIL', 'Renewal price offer'),
        (128858588, 'EMAIL', 'Renewal price offer'),
        (128858693, 'EMAIL', 'Renewal price offer'),
        (128858953, 'SMS', 'Renewal price offer'),
        (128878357, 'EMAIL', 'Renewal price offer'),
        (128939427, 'EMAIL', 'Promotion / offer'),
        (128939539, 'EMAIL', 'Promotion / offer'),
        (128940168, 'EMAIL', 'Renewal price offer'),
        (128940259, 'EMAIL', 'Renewal price offer'),
        (128953488, 'EMAIL', 'Renewal price offer'),
        (128972460, 'EMAIL', 'Renewal price offer'),
        (128989223, 'EMAIL', 'Renewal price offer'),
        (128989527, 'EMAIL', 'Renewal price offer'),
        (129045465, 'EMAIL', 'Renewal price offer'),
        (129058271, 'EMAIL', 'Renewal price offer'),
        (129071958, 'EMAIL', 'Renewal price offer'),
        (129073553, 'EMAIL', 'Promotion / offer'),
        (129085317, 'EMAIL', 'Renewal price offer'),
        (129085387, 'SMS', 'Renewal price offer'),
        (129143806, 'EMAIL', 'Renewal price offer'),
        (129156671, 'EMAIL', 'Renewal price offer'),
        (129760550, 'EMAIL', 'Promotion / offer'),
        (129869157, 'EMAIL', 'Promotion / offer'),
        (129988244, 'EMAIL', 'Promotion / offer'),
        (132056543, 'EMAIL', 'Promotion / offer'),
        (132463711, 'SMS', 'Renewal price offer'),
        (132480110, 'EMAIL', 'Promotion / offer'),
        (132965348, 'EMAIL', 'Promotion / offer'),
        (133009894, 'EMAIL', 'Promotion / offer'),
        (133247530, 'EMAIL', 'Winback'),
        (133307696, 'EMAIL', 'Winback'),
        (133307696, 'SMS', 'Winback'),
        (133348840, 'EMAIL', 'Promotion / offer'),
        (133446184, 'EMAIL', 'Winback'),
        (133446184, 'SMS', 'Winback'),
        (133466737, 'EMAIL', 'Winback'),
        (133484444, 'EMAIL', 'Winback'),
        (133484733, 'EMAIL', 'Winback'),
        (133507886, 'EMAIL', 'Winback'),
        (133507886, 'SMS', 'Winback'),
        (133644309, 'EMAIL', 'Winback'),
        (133644309, 'SMS', 'Winback'),
        (133646131, 'EMAIL', 'Winback'),
        (133646131, 'SMS', 'Winback')
         as t(CampaignId, Channel, OfferClass)) o
        on  o.CampaignId = h.campaign_id
        and o.Channel = coalesce(h.Channel, '')
where   h.SCV_Customer_Key is not null
  and   (h.SentCount > 0 or h.ClickCount > 0)


## beh_offer_row


In [ ]:
/****************************************************************************************************/
/*  Name        : stg.beh_offer_row                                                                 */
/*  Description : Offer rows for the rolling windows: the send (IsSend, IsClickedSend) on the send  */
/*                date, and for a clicked offer the first new-business sale and the first renewal   */
/*                of the customer within 30 days of the click, each on its own date (IsNbFollow,    */
/*                IsRenewalFollow), so a follow-through counts only once it has happened.           */
/*  Reads       : stg.beh_offer_msg, ods_v2.PolicySale, stg.beh_policy_scv                          */
/****************************************************************************************************/
create or replace table ${beh.stg}.beh_offer_row as
select  SCV_Customer_Key, SendDate as ActivityDate, 1 as IsSend,
        case when ClickDate is not null then 1 else 0 end as IsClickedSend, 0 as IsNbFollow, 0 as IsRenewalFollow
from    ${beh.stg}.beh_offer_msg
union all
select  m.SCV_Customer_Key, min(s.TxDate), 0, 0,
        max(case when s.SaleType = 'New Business' then 1 else 0 end),
        max(case when s.SaleType = 'Renewals' then 1 else 0 end)
from    ${beh.stg}.beh_offer_msg m
join    (select l.SCV_Customer_Key, p.ReportingSaleDate as TxDate, p.ReportingSaleType as SaleType
         from   ${beh.ods}.policysale p
         join   ${beh.stg}.beh_policy_scv l on l.PolicyCode = p.PolicyCode
         where  p.ReportingSaleType in ('New Business', 'Renewals')) s
        on  s.SCV_Customer_Key = m.SCV_Customer_Key
        and s.TxDate between m.ClickDate and date_add(m.ClickDate, 30)
where   m.ClickDate is not null
group by m.CH_Id, m.SCV_Customer_Key, s.SaleType
having  min(s.TxDate) <= to_date('${beh.last_me}')


## beh_friction_v3


In [ ]:
/****************************************************************************************************/
/*  Name        : stg.beh_friction_v3                                                               */
/*  Description : Friction to John's spec v3.3 (agreed 3 Nov 2026, run note section 12): one row    */
/*                per completed episode of the friction extract (setting friction_v3_table), with   */
/*                the number of v3.3 friction measures the customer was eligible for, affected by,  */
/*                with a known resolution and resolved. An episode counts on its end date, and      */
/*                only once it has ended by the extract's data date and by last_me, so an episode   */
/*                still open (a renewal not yet decided) is never read. Renewal episodes end at     */
/*                the renewal decision.                                                             */
/*  Reads       : ods.friction_episode_v3_<data date>                                               */
/*  Key         : EpisodeID                                                                         */
/****************************************************************************************************/
create or replace table ${beh.stg}.beh_friction_v3 as
select  f.EpisodeID,
        f.scv_id                                                as SCV_Customer_Key,
        upper(trim(f.PolicyRef))                                as PolicyCode,
        f.Product,
        f.DueDate,
        to_date(f.EpisodeEnd)                                   as ActivityDate,
        case when f.EpisodeType = 'Renewal' then 0 else 1 end   as InLife,
        case f.EpisodeType when 'Renewal' then 'renewal' when 'First year' then 'firstyear' when 'Change' then 'change' when 'Document request' then 'docreq' end                                             as EpisodeGroup,
        coalesce(cast(f.Car_FirstYear_F1_SalesCallWait_Eligible as int), 0)
      + coalesce(cast(f.Car_FirstYear_F2_StartedOnlineFinishedByPhone_Eligible as int), 0)
      + coalesce(cast(f.Car_FirstYear_F3_NoPriceOnline_Eligible as int), 0)
      + coalesce(cast(f.Car_FirstYear_F4_DocsChasedByCustomer_Eligible as int), 0)
      + coalesce(cast(f.Car_FirstYear_F5_LoanSetupChase_Eligible as int), 0)
      + coalesce(cast(f.Car_Renewal_R1_FailedLogin_Eligible as int), 0)
      + coalesce(cast(f.Car_Renewal_R2_RenewalCallWait_Eligible as int), 0)
      + coalesce(cast(f.Car_Renewal_R3_OnlineThenCalled_Eligible as int), 0)
      + coalesce(cast(f.Car_Renewal_R4_RequoteNoPrice_Eligible as int), 0)
      + coalesce(cast(f.Car_Renewal_R5_RenewalHeld_Eligible as int), 0)
      + coalesce(cast(f.Car_Renewal_R6_RenewalDateMissed_Eligible as int), 0)
      + coalesce(cast(f.Car_Change_C1_ChangeCallWait_Eligible as int), 0)
      + coalesce(cast(f.Car_DocRequest_C3_DocRequestCallWait_Eligible as int), 0)
      + coalesce(cast(f.Car_DocRequest_C4_DocNotArrived_Eligible as int), 0)
      + coalesce(cast(f.Car_DocRequest_C5_DocFromInsurer_Eligible as int), 0)
      + coalesce(cast(f.Home_FirstYear_F1_SalesCallWait_Eligible as int), 0)
      + coalesce(cast(f.Home_FirstYear_F2_StartedOnlineFinishedByPhone_Eligible as int), 0)
      + coalesce(cast(f.Home_FirstYear_F3_NoPriceOnline_Eligible as int), 0)
      + coalesce(cast(f.Home_FirstYear_F4_DocsChasedByCustomer_Eligible as int), 0)
      + coalesce(cast(f.Home_FirstYear_F5_LoanSetupChase_Eligible as int), 0)
      + coalesce(cast(f.Home_Renewal_R1_FailedLogin_Eligible as int), 0)
      + coalesce(cast(f.Home_Renewal_R2_RenewalCallWait_Eligible as int), 0)
      + coalesce(cast(f.Home_Renewal_R3_OnlineThenCalled_Eligible as int), 0)
      + coalesce(cast(f.Home_Renewal_R4_RequoteNoPrice_Eligible as int), 0)
      + coalesce(cast(f.Home_Renewal_R5_RenewalHeld_Eligible as int), 0)
      + coalesce(cast(f.Home_Renewal_R6_RenewalDateMissed_Eligible as int), 0)
      + coalesce(cast(f.Home_Change_C1_ChangeCallWait_Eligible as int), 0)
      + coalesce(cast(f.Home_DocRequest_C3_DocRequestCallWait_Eligible as int), 0)
      + coalesce(cast(f.Home_DocRequest_C4_DocNotArrived_Eligible as int), 0)
      + coalesce(cast(f.Home_DocRequest_C5_DocFromInsurer_Eligible as int), 0)
      + coalesce(cast(f.Van_FirstYear_F1_SalesCallWait_Eligible as int), 0)
      + coalesce(cast(f.Van_FirstYear_F2_StartedOnlineFinishedByPhone_Eligible as int), 0)
      + coalesce(cast(f.Van_FirstYear_F3_NoPriceOnline_Eligible as int), 0)
      + coalesce(cast(f.Van_FirstYear_F4_DocsChasedByCustomer_Eligible as int), 0)
      + coalesce(cast(f.Van_FirstYear_F5_LoanSetupChase_Eligible as int), 0)
      + coalesce(cast(f.Van_Renewal_R1_FailedLogin_Eligible as int), 0)
      + coalesce(cast(f.Van_Renewal_R2_RenewalCallWait_Eligible as int), 0)
      + coalesce(cast(f.Van_Renewal_R3_OnlineThenCalled_Eligible as int), 0)
      + coalesce(cast(f.Van_Renewal_R4_RequoteNoPrice_Eligible as int), 0)
      + coalesce(cast(f.Van_Renewal_R5_RenewalHeld_Eligible as int), 0)
      + coalesce(cast(f.Van_Renewal_R6_RenewalDateMissed_Eligible as int), 0)
      + coalesce(cast(f.Van_Change_C1_ChangeCallWait_Eligible as int), 0)
      + coalesce(cast(f.Van_DocRequest_C3_DocRequestCallWait_Eligible as int), 0)
      + coalesce(cast(f.Van_DocRequest_C4_DocNotArrived_Eligible as int), 0)
      + coalesce(cast(f.Van_DocRequest_C5_DocFromInsurer_Eligible as int), 0)                                          as Eligible,
        case when f.Car_FirstYear_F1_SalesCallWait_Eligible = 1 and f.Car_FirstYear_F1_SalesCallWait_Affected = 1 then 1 else 0 end
      + case when f.Car_FirstYear_F2_StartedOnlineFinishedByPhone_Eligible = 1 and f.Car_FirstYear_F2_StartedOnlineFinishedByPhone_Affected = 1 then 1 else 0 end
      + case when f.Car_FirstYear_F3_NoPriceOnline_Eligible = 1 and f.Car_FirstYear_F3_NoPriceOnline_Affected = 1 then 1 else 0 end
      + case when f.Car_FirstYear_F4_DocsChasedByCustomer_Eligible = 1 and f.Car_FirstYear_F4_DocsChasedByCustomer_Affected = 1 then 1 else 0 end
      + case when f.Car_FirstYear_F5_LoanSetupChase_Eligible = 1 and f.Car_FirstYear_F5_LoanSetupChase_Affected = 1 then 1 else 0 end
      + case when f.Car_Renewal_R1_FailedLogin_Eligible = 1 and f.Car_Renewal_R1_FailedLogin_Affected = 1 then 1 else 0 end
      + case when f.Car_Renewal_R2_RenewalCallWait_Eligible = 1 and f.Car_Renewal_R2_RenewalCallWait_Affected = 1 then 1 else 0 end
      + case when f.Car_Renewal_R3_OnlineThenCalled_Eligible = 1 and f.Car_Renewal_R3_OnlineThenCalled_Affected = 1 then 1 else 0 end
      + case when f.Car_Renewal_R4_RequoteNoPrice_Eligible = 1 and f.Car_Renewal_R4_RequoteNoPrice_Affected = 1 then 1 else 0 end
      + case when f.Car_Renewal_R5_RenewalHeld_Eligible = 1 and f.Car_Renewal_R5_RenewalHeld_Affected = 1 then 1 else 0 end
      + case when f.Car_Renewal_R6_RenewalDateMissed_Eligible = 1 and f.Car_Renewal_R6_RenewalDateMissed_Affected = 1 then 1 else 0 end
      + case when f.Car_Change_C1_ChangeCallWait_Eligible = 1 and f.Car_Change_C1_ChangeCallWait_Affected = 1 then 1 else 0 end
      + case when f.Car_DocRequest_C3_DocRequestCallWait_Eligible = 1 and f.Car_DocRequest_C3_DocRequestCallWait_Affected = 1 then 1 else 0 end
      + case when f.Car_DocRequest_C4_DocNotArrived_Eligible = 1 and f.Car_DocRequest_C4_DocNotArrived_Affected = 1 then 1 else 0 end
      + case when f.Car_DocRequest_C5_DocFromInsurer_Eligible = 1 and f.Car_DocRequest_C5_DocFromInsurer_Affected = 1 then 1 else 0 end
      + case when f.Home_FirstYear_F1_SalesCallWait_Eligible = 1 and f.Home_FirstYear_F1_SalesCallWait_Affected = 1 then 1 else 0 end
      + case when f.Home_FirstYear_F2_StartedOnlineFinishedByPhone_Eligible = 1 and f.Home_FirstYear_F2_StartedOnlineFinishedByPhone_Affected = 1 then 1 else 0 end
      + case when f.Home_FirstYear_F3_NoPriceOnline_Eligible = 1 and f.Home_FirstYear_F3_NoPriceOnline_Affected = 1 then 1 else 0 end
      + case when f.Home_FirstYear_F4_DocsChasedByCustomer_Eligible = 1 and f.Home_FirstYear_F4_DocsChasedByCustomer_Affected = 1 then 1 else 0 end
      + case when f.Home_FirstYear_F5_LoanSetupChase_Eligible = 1 and f.Home_FirstYear_F5_LoanSetupChase_Affected = 1 then 1 else 0 end
      + case when f.Home_Renewal_R1_FailedLogin_Eligible = 1 and f.Home_Renewal_R1_FailedLogin_Affected = 1 then 1 else 0 end
      + case when f.Home_Renewal_R2_RenewalCallWait_Eligible = 1 and f.Home_Renewal_R2_RenewalCallWait_Affected = 1 then 1 else 0 end
      + case when f.Home_Renewal_R3_OnlineThenCalled_Eligible = 1 and f.Home_Renewal_R3_OnlineThenCalled_Affected = 1 then 1 else 0 end
      + case when f.Home_Renewal_R4_RequoteNoPrice_Eligible = 1 and f.Home_Renewal_R4_RequoteNoPrice_Affected = 1 then 1 else 0 end
      + case when f.Home_Renewal_R5_RenewalHeld_Eligible = 1 and f.Home_Renewal_R5_RenewalHeld_Affected = 1 then 1 else 0 end
      + case when f.Home_Renewal_R6_RenewalDateMissed_Eligible = 1 and f.Home_Renewal_R6_RenewalDateMissed_Affected = 1 then 1 else 0 end
      + case when f.Home_Change_C1_ChangeCallWait_Eligible = 1 and f.Home_Change_C1_ChangeCallWait_Affected = 1 then 1 else 0 end
      + case when f.Home_DocRequest_C3_DocRequestCallWait_Eligible = 1 and f.Home_DocRequest_C3_DocRequestCallWait_Affected = 1 then 1 else 0 end
      + case when f.Home_DocRequest_C4_DocNotArrived_Eligible = 1 and f.Home_DocRequest_C4_DocNotArrived_Affected = 1 then 1 else 0 end
      + case when f.Home_DocRequest_C5_DocFromInsurer_Eligible = 1 and f.Home_DocRequest_C5_DocFromInsurer_Affected = 1 then 1 else 0 end
      + case when f.Van_FirstYear_F1_SalesCallWait_Eligible = 1 and f.Van_FirstYear_F1_SalesCallWait_Affected = 1 then 1 else 0 end
      + case when f.Van_FirstYear_F2_StartedOnlineFinishedByPhone_Eligible = 1 and f.Van_FirstYear_F2_StartedOnlineFinishedByPhone_Affected = 1 then 1 else 0 end
      + case when f.Van_FirstYear_F3_NoPriceOnline_Eligible = 1 and f.Van_FirstYear_F3_NoPriceOnline_Affected = 1 then 1 else 0 end
      + case when f.Van_FirstYear_F4_DocsChasedByCustomer_Eligible = 1 and f.Van_FirstYear_F4_DocsChasedByCustomer_Affected = 1 then 1 else 0 end
      + case when f.Van_FirstYear_F5_LoanSetupChase_Eligible = 1 and f.Van_FirstYear_F5_LoanSetupChase_Affected = 1 then 1 else 0 end
      + case when f.Van_Renewal_R1_FailedLogin_Eligible = 1 and f.Van_Renewal_R1_FailedLogin_Affected = 1 then 1 else 0 end
      + case when f.Van_Renewal_R2_RenewalCallWait_Eligible = 1 and f.Van_Renewal_R2_RenewalCallWait_Affected = 1 then 1 else 0 end
      + case when f.Van_Renewal_R3_OnlineThenCalled_Eligible = 1 and f.Van_Renewal_R3_OnlineThenCalled_Affected = 1 then 1 else 0 end
      + case when f.Van_Renewal_R4_RequoteNoPrice_Eligible = 1 and f.Van_Renewal_R4_RequoteNoPrice_Affected = 1 then 1 else 0 end
      + case when f.Van_Renewal_R5_RenewalHeld_Eligible = 1 and f.Van_Renewal_R5_RenewalHeld_Affected = 1 then 1 else 0 end
      + case when f.Van_Renewal_R6_RenewalDateMissed_Eligible = 1 and f.Van_Renewal_R6_RenewalDateMissed_Affected = 1 then 1 else 0 end
      + case when f.Van_Change_C1_ChangeCallWait_Eligible = 1 and f.Van_Change_C1_ChangeCallWait_Affected = 1 then 1 else 0 end
      + case when f.Van_DocRequest_C3_DocRequestCallWait_Eligible = 1 and f.Van_DocRequest_C3_DocRequestCallWait_Affected = 1 then 1 else 0 end
      + case when f.Van_DocRequest_C4_DocNotArrived_Eligible = 1 and f.Van_DocRequest_C4_DocNotArrived_Affected = 1 then 1 else 0 end
      + case when f.Van_DocRequest_C5_DocFromInsurer_Eligible = 1 and f.Van_DocRequest_C5_DocFromInsurer_Affected = 1 then 1 else 0 end                                          as Affected,
        case when f.Car_FirstYear_F1_SalesCallWait_Affected = 1 and f.Car_FirstYear_F1_SalesCallWait_Resolution in ('Resolved', 'Unresolved') then 1 else 0 end
      + case when f.Car_FirstYear_F4_DocsChasedByCustomer_Affected = 1 and f.Car_FirstYear_F4_DocsChasedByCustomer_Resolution in ('Resolved', 'Unresolved') then 1 else 0 end
      + case when f.Car_FirstYear_F5_LoanSetupChase_Affected = 1 and f.Car_FirstYear_F5_LoanSetupChase_Resolution in ('Resolved', 'Unresolved') then 1 else 0 end
      + case when f.Car_Renewal_R1_FailedLogin_Affected = 1 and f.Car_Renewal_R1_FailedLogin_Resolution in ('Resolved', 'Unresolved') then 1 else 0 end
      + case when f.Car_Renewal_R2_RenewalCallWait_Affected = 1 and f.Car_Renewal_R2_RenewalCallWait_Resolution in ('Resolved', 'Unresolved') then 1 else 0 end
      + case when f.Car_Renewal_R4_RequoteNoPrice_Affected = 1 and f.Car_Renewal_R4_RequoteNoPrice_Resolution in ('Resolved', 'Unresolved') then 1 else 0 end
      + case when f.Car_Renewal_R5_RenewalHeld_Affected = 1 and f.Car_Renewal_R5_RenewalHeld_Resolution in ('Resolved', 'Unresolved') then 1 else 0 end
      + case when f.Car_Change_C1_ChangeCallWait_Affected = 1 and f.Car_Change_C1_ChangeCallWait_Resolution in ('Resolved', 'Unresolved') then 1 else 0 end
      + case when f.Car_DocRequest_C3_DocRequestCallWait_Affected = 1 and f.Car_DocRequest_C3_DocRequestCallWait_Resolution in ('Resolved', 'Unresolved') then 1 else 0 end
      + case when f.Car_DocRequest_C4_DocNotArrived_Affected = 1 and f.Car_DocRequest_C4_DocNotArrived_Resolution in ('Resolved', 'Unresolved') then 1 else 0 end
      + case when f.Car_DocRequest_C5_DocFromInsurer_Affected = 1 and f.Car_DocRequest_C5_DocFromInsurer_Resolution in ('Resolved', 'Unresolved') then 1 else 0 end
      + case when f.Home_FirstYear_F1_SalesCallWait_Affected = 1 and f.Home_FirstYear_F1_SalesCallWait_Resolution in ('Resolved', 'Unresolved') then 1 else 0 end
      + case when f.Home_FirstYear_F4_DocsChasedByCustomer_Affected = 1 and f.Home_FirstYear_F4_DocsChasedByCustomer_Resolution in ('Resolved', 'Unresolved') then 1 else 0 end
      + case when f.Home_FirstYear_F5_LoanSetupChase_Affected = 1 and f.Home_FirstYear_F5_LoanSetupChase_Resolution in ('Resolved', 'Unresolved') then 1 else 0 end
      + case when f.Home_Renewal_R1_FailedLogin_Affected = 1 and f.Home_Renewal_R1_FailedLogin_Resolution in ('Resolved', 'Unresolved') then 1 else 0 end
      + case when f.Home_Renewal_R2_RenewalCallWait_Affected = 1 and f.Home_Renewal_R2_RenewalCallWait_Resolution in ('Resolved', 'Unresolved') then 1 else 0 end
      + case when f.Home_Renewal_R4_RequoteNoPrice_Affected = 1 and f.Home_Renewal_R4_RequoteNoPrice_Resolution in ('Resolved', 'Unresolved') then 1 else 0 end
      + case when f.Home_Renewal_R5_RenewalHeld_Affected = 1 and f.Home_Renewal_R5_RenewalHeld_Resolution in ('Resolved', 'Unresolved') then 1 else 0 end
      + case when f.Home_Change_C1_ChangeCallWait_Affected = 1 and f.Home_Change_C1_ChangeCallWait_Resolution in ('Resolved', 'Unresolved') then 1 else 0 end
      + case when f.Home_DocRequest_C3_DocRequestCallWait_Affected = 1 and f.Home_DocRequest_C3_DocRequestCallWait_Resolution in ('Resolved', 'Unresolved') then 1 else 0 end
      + case when f.Home_DocRequest_C4_DocNotArrived_Affected = 1 and f.Home_DocRequest_C4_DocNotArrived_Resolution in ('Resolved', 'Unresolved') then 1 else 0 end
      + case when f.Home_DocRequest_C5_DocFromInsurer_Affected = 1 and f.Home_DocRequest_C5_DocFromInsurer_Resolution in ('Resolved', 'Unresolved') then 1 else 0 end
      + case when f.Van_FirstYear_F1_SalesCallWait_Affected = 1 and f.Van_FirstYear_F1_SalesCallWait_Resolution in ('Resolved', 'Unresolved') then 1 else 0 end
      + case when f.Van_FirstYear_F4_DocsChasedByCustomer_Affected = 1 and f.Van_FirstYear_F4_DocsChasedByCustomer_Resolution in ('Resolved', 'Unresolved') then 1 else 0 end
      + case when f.Van_FirstYear_F5_LoanSetupChase_Affected = 1 and f.Van_FirstYear_F5_LoanSetupChase_Resolution in ('Resolved', 'Unresolved') then 1 else 0 end
      + case when f.Van_Renewal_R1_FailedLogin_Affected = 1 and f.Van_Renewal_R1_FailedLogin_Resolution in ('Resolved', 'Unresolved') then 1 else 0 end
      + case when f.Van_Renewal_R2_RenewalCallWait_Affected = 1 and f.Van_Renewal_R2_RenewalCallWait_Resolution in ('Resolved', 'Unresolved') then 1 else 0 end
      + case when f.Van_Renewal_R4_RequoteNoPrice_Affected = 1 and f.Van_Renewal_R4_RequoteNoPrice_Resolution in ('Resolved', 'Unresolved') then 1 else 0 end
      + case when f.Van_Renewal_R5_RenewalHeld_Affected = 1 and f.Van_Renewal_R5_RenewalHeld_Resolution in ('Resolved', 'Unresolved') then 1 else 0 end
      + case when f.Van_Change_C1_ChangeCallWait_Affected = 1 and f.Van_Change_C1_ChangeCallWait_Resolution in ('Resolved', 'Unresolved') then 1 else 0 end
      + case when f.Van_DocRequest_C3_DocRequestCallWait_Affected = 1 and f.Van_DocRequest_C3_DocRequestCallWait_Resolution in ('Resolved', 'Unresolved') then 1 else 0 end
      + case when f.Van_DocRequest_C4_DocNotArrived_Affected = 1 and f.Van_DocRequest_C4_DocNotArrived_Resolution in ('Resolved', 'Unresolved') then 1 else 0 end
      + case when f.Van_DocRequest_C5_DocFromInsurer_Affected = 1 and f.Van_DocRequest_C5_DocFromInsurer_Resolution in ('Resolved', 'Unresolved') then 1 else 0 end                                         as ResKnown,
        case when f.Car_FirstYear_F1_SalesCallWait_Affected = 1 and f.Car_FirstYear_F1_SalesCallWait_Resolution = 'Resolved' then 1 else 0 end
      + case when f.Car_FirstYear_F4_DocsChasedByCustomer_Affected = 1 and f.Car_FirstYear_F4_DocsChasedByCustomer_Resolution = 'Resolved' then 1 else 0 end
      + case when f.Car_FirstYear_F5_LoanSetupChase_Affected = 1 and f.Car_FirstYear_F5_LoanSetupChase_Resolution = 'Resolved' then 1 else 0 end
      + case when f.Car_Renewal_R1_FailedLogin_Affected = 1 and f.Car_Renewal_R1_FailedLogin_Resolution = 'Resolved' then 1 else 0 end
      + case when f.Car_Renewal_R2_RenewalCallWait_Affected = 1 and f.Car_Renewal_R2_RenewalCallWait_Resolution = 'Resolved' then 1 else 0 end
      + case when f.Car_Renewal_R4_RequoteNoPrice_Affected = 1 and f.Car_Renewal_R4_RequoteNoPrice_Resolution = 'Resolved' then 1 else 0 end
      + case when f.Car_Renewal_R5_RenewalHeld_Affected = 1 and f.Car_Renewal_R5_RenewalHeld_Resolution = 'Resolved' then 1 else 0 end
      + case when f.Car_Change_C1_ChangeCallWait_Affected = 1 and f.Car_Change_C1_ChangeCallWait_Resolution = 'Resolved' then 1 else 0 end
      + case when f.Car_DocRequest_C3_DocRequestCallWait_Affected = 1 and f.Car_DocRequest_C3_DocRequestCallWait_Resolution = 'Resolved' then 1 else 0 end
      + case when f.Car_DocRequest_C4_DocNotArrived_Affected = 1 and f.Car_DocRequest_C4_DocNotArrived_Resolution = 'Resolved' then 1 else 0 end
      + case when f.Car_DocRequest_C5_DocFromInsurer_Affected = 1 and f.Car_DocRequest_C5_DocFromInsurer_Resolution = 'Resolved' then 1 else 0 end
      + case when f.Home_FirstYear_F1_SalesCallWait_Affected = 1 and f.Home_FirstYear_F1_SalesCallWait_Resolution = 'Resolved' then 1 else 0 end
      + case when f.Home_FirstYear_F4_DocsChasedByCustomer_Affected = 1 and f.Home_FirstYear_F4_DocsChasedByCustomer_Resolution = 'Resolved' then 1 else 0 end
      + case when f.Home_FirstYear_F5_LoanSetupChase_Affected = 1 and f.Home_FirstYear_F5_LoanSetupChase_Resolution = 'Resolved' then 1 else 0 end
      + case when f.Home_Renewal_R1_FailedLogin_Affected = 1 and f.Home_Renewal_R1_FailedLogin_Resolution = 'Resolved' then 1 else 0 end
      + case when f.Home_Renewal_R2_RenewalCallWait_Affected = 1 and f.Home_Renewal_R2_RenewalCallWait_Resolution = 'Resolved' then 1 else 0 end
      + case when f.Home_Renewal_R4_RequoteNoPrice_Affected = 1 and f.Home_Renewal_R4_RequoteNoPrice_Resolution = 'Resolved' then 1 else 0 end
      + case when f.Home_Renewal_R5_RenewalHeld_Affected = 1 and f.Home_Renewal_R5_RenewalHeld_Resolution = 'Resolved' then 1 else 0 end
      + case when f.Home_Change_C1_ChangeCallWait_Affected = 1 and f.Home_Change_C1_ChangeCallWait_Resolution = 'Resolved' then 1 else 0 end
      + case when f.Home_DocRequest_C3_DocRequestCallWait_Affected = 1 and f.Home_DocRequest_C3_DocRequestCallWait_Resolution = 'Resolved' then 1 else 0 end
      + case when f.Home_DocRequest_C4_DocNotArrived_Affected = 1 and f.Home_DocRequest_C4_DocNotArrived_Resolution = 'Resolved' then 1 else 0 end
      + case when f.Home_DocRequest_C5_DocFromInsurer_Affected = 1 and f.Home_DocRequest_C5_DocFromInsurer_Resolution = 'Resolved' then 1 else 0 end
      + case when f.Van_FirstYear_F1_SalesCallWait_Affected = 1 and f.Van_FirstYear_F1_SalesCallWait_Resolution = 'Resolved' then 1 else 0 end
      + case when f.Van_FirstYear_F4_DocsChasedByCustomer_Affected = 1 and f.Van_FirstYear_F4_DocsChasedByCustomer_Resolution = 'Resolved' then 1 else 0 end
      + case when f.Van_FirstYear_F5_LoanSetupChase_Affected = 1 and f.Van_FirstYear_F5_LoanSetupChase_Resolution = 'Resolved' then 1 else 0 end
      + case when f.Van_Renewal_R1_FailedLogin_Affected = 1 and f.Van_Renewal_R1_FailedLogin_Resolution = 'Resolved' then 1 else 0 end
      + case when f.Van_Renewal_R2_RenewalCallWait_Affected = 1 and f.Van_Renewal_R2_RenewalCallWait_Resolution = 'Resolved' then 1 else 0 end
      + case when f.Van_Renewal_R4_RequoteNoPrice_Affected = 1 and f.Van_Renewal_R4_RequoteNoPrice_Resolution = 'Resolved' then 1 else 0 end
      + case when f.Van_Renewal_R5_RenewalHeld_Affected = 1 and f.Van_Renewal_R5_RenewalHeld_Resolution = 'Resolved' then 1 else 0 end
      + case when f.Van_Change_C1_ChangeCallWait_Affected = 1 and f.Van_Change_C1_ChangeCallWait_Resolution = 'Resolved' then 1 else 0 end
      + case when f.Van_DocRequest_C3_DocRequestCallWait_Affected = 1 and f.Van_DocRequest_C3_DocRequestCallWait_Resolution = 'Resolved' then 1 else 0 end
      + case when f.Van_DocRequest_C4_DocNotArrived_Affected = 1 and f.Van_DocRequest_C4_DocNotArrived_Resolution = 'Resolved' then 1 else 0 end
      + case when f.Van_DocRequest_C5_DocFromInsurer_Affected = 1 and f.Van_DocRequest_C5_DocFromInsurer_Resolution = 'Resolved' then 1 else 0 end                                          as Resolved
from    ${beh.friction_v3} f
where   f.scv_id is not null
  and   f.EpisodeType in ('Renewal', 'First year', 'Change', 'Document request')
  and   to_date(f.EpisodeEnd) <= to_date(f.DataDate)
  and   to_date(f.EpisodeEnd) <= to_date('${beh.last_me}')


## beh_activity_row


In [ ]:
/****************************************************************************************************/
/*  Name        : stg.beh_activity_row                                                              */
/*  Description : Every event, episode, call, MTA, loan status change, concession, sale, document   */
/*                decision and consent statement as one row: customer, activity date, InLife (from  */
/*                the episode type, null without one) and one column per measure                    */
/*                (build/measures.py).                                                              */
/*  Reads       : stg.beh_event, stg.beh_episode_x, stg.beh_call_scv, stg.beh_mta,                  */
/*                stg.beh_pme_mta, stg.beh_loan_change, stg.beh_fee_concession, stg.beh_sale,       */
/*                stg.beh_doc_wf, stg.beh_doc_nonreceipt, stg.beh_consent, stg.beh_mfq_quote_x,     */
/*                stg.beh_web_session, stg.beh_tcall, stg.beh_friction_res, stg.beh_offer_row,      */
/*                stg.beh_friction_v3                                                               */
/****************************************************************************************************/
create or replace table ${beh.stg}.beh_activity_row as
select  e.SCV_Customer_Key as SCV_Customer_Key,
        e.ActivityDate as ActivityDate,
        cast(e.InLife as int) as InLife,
        e.prompts as prompts,
        e.ren_cust_actions as ren_cust_actions,
        e.inlife_contacts as inlife_contacts,
        e.friction_events as friction_events,
        e.doc_friction_events as doc_friction_events,
        e.chases as chases,
        0 as concessions,
        e.failed_payments as failed_payments,
        e.failed_logins as failed_logins,
        e.arrears_events as arrears_events,
        e.doc_requests as doc_requests,
        e.sales_calls as sales_calls,
        1 as events_total,
        e.online_quotes_motor as online_quotes_motor,
        e.online_quotes_home as online_quotes_home,
        e.online_quotes_van as online_quotes_van,
        e.renewal_requotes as renewal_requotes,
        0 as episodes,
        0 as episodes_friction,
        0 as episodes_completed,
        0 as calls_in,
        0 as calls_in_answered,
        0 as calls_in_abandoned,
        0 as calls_in_long_wait,
        0 as calls_in_wait_s,
        0 as calls_in_talk_s,
        0 as calls_out,
        0 as calls_out_talk_s,
        0 as mtas,
        0 as mta_records,
        0 as mtas_forced,
        0 as mtas_elective,
        0 as mta_cat_cover,
        0 as mta_cat_drivers,
        0 as mta_cat_vehicle,
        0 as mta_cat_address,
        0 as mta_cat_licence,
        0 as mta_cat_use,
        0 as mta_cat_personal,
        0 as loan_status_changes,
        0 as episodes_assisted,
        0 as episodes_channel_tagged,
        0 as channel_switches,
        0 as episodes_repeat_contact,
        0 as nb_sales,
        0 as nb_sales_web_assist,
        0 as doc_decisions,
        0 as doc_rejections,
        0 as doc_attempts,
        e.doc_nonreceipt as doc_nonreceipt,
        0 as doc_nonreceipt_premium,
        0 as consent_actions,
        0 as consent_withdrawals,
        0 as mfq_quotes,
        0 as mfq_retrievals,
        0 as mfq_insurers,
        0 as mfq_returned,
        0 as mfq_cover_compared,
        0 as mfq_cover_down,
        0 as mfq_cover_up,
        0 as mfq_device_tagged,
        0 as mfq_device_switch,
        0 as web_sessions,
        0 as web_sessions_nonquote,
        0 as web_page_views,
        0 as web_content_views,
        0 as web_other_product_sessions,
        0 as calls_other_queues,
        0 as friction_all,
        0 as friction_resolved,
        0 as friction_payment,
        0 as friction_payment_resolved,
        0 as friction_login,
        0 as friction_login_resolved,
        0 as friction_contact,
        0 as friction_contact_resolved,
        0 as friction_documents,
        0 as friction_documents_resolved,
        0 as friction_quote,
        0 as friction_quote_resolved,
        0 as friction_process,
        0 as friction_process_resolved,
        0 as friction_other,
        0 as friction_other_resolved,
        0 as offers_sent,
        0 as offers_clicked,
        0 as offer_clicks_nb,
        0 as offer_clicks_renewal,
        0 as friction_v3_eligible,
        0 as friction_v3_affected,
        0 as friction_v3_res_known,
        0 as friction_v3_resolved,
        0 as friction_v3_renewal_eligible,
        0 as friction_v3_renewal_affected,
        0 as friction_v3_renewal_res_known,
        0 as friction_v3_renewal_resolved,
        0 as friction_v3_firstyear_eligible,
        0 as friction_v3_firstyear_affected,
        0 as friction_v3_firstyear_res_known,
        0 as friction_v3_firstyear_resolved,
        0 as friction_v3_change_eligible,
        0 as friction_v3_change_affected,
        0 as friction_v3_change_res_known,
        0 as friction_v3_change_resolved,
        0 as friction_v3_docreq_eligible,
        0 as friction_v3_docreq_affected,
        0 as friction_v3_docreq_res_known,
        0 as friction_v3_docreq_resolved
from    ${beh.stg}.beh_event e
union all
select  p.SCV_Customer_Key as SCV_Customer_Key,
        p.EpisodeEndDate as ActivityDate,
        cast(p.InLife as int) as InLife,
        0 as prompts,
        0 as ren_cust_actions,
        0 as inlife_contacts,
        0 as friction_events,
        0 as doc_friction_events,
        0 as chases,
        0 as concessions,
        0 as failed_payments,
        0 as failed_logins,
        0 as arrears_events,
        0 as doc_requests,
        0 as sales_calls,
        0 as events_total,
        0 as online_quotes_motor,
        0 as online_quotes_home,
        0 as online_quotes_van,
        0 as renewal_requotes,
        1 as episodes,
        p.HasFriction as episodes_friction,
        p.HasOutcome as episodes_completed,
        0 as calls_in,
        0 as calls_in_answered,
        0 as calls_in_abandoned,
        0 as calls_in_long_wait,
        0 as calls_in_wait_s,
        0 as calls_in_talk_s,
        0 as calls_out,
        0 as calls_out_talk_s,
        0 as mtas,
        0 as mta_records,
        0 as mtas_forced,
        0 as mtas_elective,
        0 as mta_cat_cover,
        0 as mta_cat_drivers,
        0 as mta_cat_vehicle,
        0 as mta_cat_address,
        0 as mta_cat_licence,
        0 as mta_cat_use,
        0 as mta_cat_personal,
        0 as loan_status_changes,
        p.HasAssisted as episodes_assisted,
        p.HasChannelStep as episodes_channel_tagged,
        p.ChannelSwitches as channel_switches,
        p.RepeatContact as episodes_repeat_contact,
        0 as nb_sales,
        0 as nb_sales_web_assist,
        0 as doc_decisions,
        0 as doc_rejections,
        0 as doc_attempts,
        0 as doc_nonreceipt,
        0 as doc_nonreceipt_premium,
        0 as consent_actions,
        0 as consent_withdrawals,
        0 as mfq_quotes,
        0 as mfq_retrievals,
        0 as mfq_insurers,
        0 as mfq_returned,
        0 as mfq_cover_compared,
        0 as mfq_cover_down,
        0 as mfq_cover_up,
        0 as mfq_device_tagged,
        0 as mfq_device_switch,
        0 as web_sessions,
        0 as web_sessions_nonquote,
        0 as web_page_views,
        0 as web_content_views,
        0 as web_other_product_sessions,
        0 as calls_other_queues,
        0 as friction_all,
        0 as friction_resolved,
        0 as friction_payment,
        0 as friction_payment_resolved,
        0 as friction_login,
        0 as friction_login_resolved,
        0 as friction_contact,
        0 as friction_contact_resolved,
        0 as friction_documents,
        0 as friction_documents_resolved,
        0 as friction_quote,
        0 as friction_quote_resolved,
        0 as friction_process,
        0 as friction_process_resolved,
        0 as friction_other,
        0 as friction_other_resolved,
        0 as offers_sent,
        0 as offers_clicked,
        0 as offer_clicks_nb,
        0 as offer_clicks_renewal,
        0 as friction_v3_eligible,
        0 as friction_v3_affected,
        0 as friction_v3_res_known,
        0 as friction_v3_resolved,
        0 as friction_v3_renewal_eligible,
        0 as friction_v3_renewal_affected,
        0 as friction_v3_renewal_res_known,
        0 as friction_v3_renewal_resolved,
        0 as friction_v3_firstyear_eligible,
        0 as friction_v3_firstyear_affected,
        0 as friction_v3_firstyear_res_known,
        0 as friction_v3_firstyear_resolved,
        0 as friction_v3_change_eligible,
        0 as friction_v3_change_affected,
        0 as friction_v3_change_res_known,
        0 as friction_v3_change_resolved,
        0 as friction_v3_docreq_eligible,
        0 as friction_v3_docreq_affected,
        0 as friction_v3_docreq_res_known,
        0 as friction_v3_docreq_resolved
from    ${beh.stg}.beh_episode_x p
union all
select  c.SCV_Customer_Key as SCV_Customer_Key,
        c.CallDate as ActivityDate,
        cast(null as int) as InLife,
        0 as prompts,
        0 as ren_cust_actions,
        0 as inlife_contacts,
        0 as friction_events,
        0 as doc_friction_events,
        0 as chases,
        0 as concessions,
        0 as failed_payments,
        0 as failed_logins,
        0 as arrears_events,
        0 as doc_requests,
        0 as sales_calls,
        0 as events_total,
        0 as online_quotes_motor,
        0 as online_quotes_home,
        0 as online_quotes_van,
        0 as renewal_requotes,
        0 as episodes,
        0 as episodes_friction,
        0 as episodes_completed,
        case when c.Direction = 'inbound' then 1 else 0 end as calls_in,
        case when c.Direction = 'inbound' and c.AgentAnswered > 0 then 1 else 0 end as calls_in_answered,
        case when c.Direction = 'inbound' and c.Abandoned > 0 then 1 else 0 end as calls_in_abandoned,
        case when c.Direction = 'inbound' and (c.Abandoned > 0 or c.WaitMs > 600000) then 1 else 0 end as calls_in_long_wait,
        case when c.Direction = 'inbound' then c.WaitMs / 1000.0 else 0 end as calls_in_wait_s,
        case when c.Direction = 'inbound' then c.TalkMs / 1000.0 else 0 end as calls_in_talk_s,
        case when c.Direction = 'outbound' then 1 else 0 end as calls_out,
        case when c.Direction = 'outbound' then c.TalkMs / 1000.0 else 0 end as calls_out_talk_s,
        0 as mtas,
        0 as mta_records,
        0 as mtas_forced,
        0 as mtas_elective,
        0 as mta_cat_cover,
        0 as mta_cat_drivers,
        0 as mta_cat_vehicle,
        0 as mta_cat_address,
        0 as mta_cat_licence,
        0 as mta_cat_use,
        0 as mta_cat_personal,
        0 as loan_status_changes,
        0 as episodes_assisted,
        0 as episodes_channel_tagged,
        0 as channel_switches,
        0 as episodes_repeat_contact,
        0 as nb_sales,
        0 as nb_sales_web_assist,
        0 as doc_decisions,
        0 as doc_rejections,
        0 as doc_attempts,
        0 as doc_nonreceipt,
        0 as doc_nonreceipt_premium,
        0 as consent_actions,
        0 as consent_withdrawals,
        0 as mfq_quotes,
        0 as mfq_retrievals,
        0 as mfq_insurers,
        0 as mfq_returned,
        0 as mfq_cover_compared,
        0 as mfq_cover_down,
        0 as mfq_cover_up,
        0 as mfq_device_tagged,
        0 as mfq_device_switch,
        0 as web_sessions,
        0 as web_sessions_nonquote,
        0 as web_page_views,
        0 as web_content_views,
        0 as web_other_product_sessions,
        0 as calls_other_queues,
        0 as friction_all,
        0 as friction_resolved,
        0 as friction_payment,
        0 as friction_payment_resolved,
        0 as friction_login,
        0 as friction_login_resolved,
        0 as friction_contact,
        0 as friction_contact_resolved,
        0 as friction_documents,
        0 as friction_documents_resolved,
        0 as friction_quote,
        0 as friction_quote_resolved,
        0 as friction_process,
        0 as friction_process_resolved,
        0 as friction_other,
        0 as friction_other_resolved,
        0 as offers_sent,
        0 as offers_clicked,
        0 as offer_clicks_nb,
        0 as offer_clicks_renewal,
        0 as friction_v3_eligible,
        0 as friction_v3_affected,
        0 as friction_v3_res_known,
        0 as friction_v3_resolved,
        0 as friction_v3_renewal_eligible,
        0 as friction_v3_renewal_affected,
        0 as friction_v3_renewal_res_known,
        0 as friction_v3_renewal_resolved,
        0 as friction_v3_firstyear_eligible,
        0 as friction_v3_firstyear_affected,
        0 as friction_v3_firstyear_res_known,
        0 as friction_v3_firstyear_resolved,
        0 as friction_v3_change_eligible,
        0 as friction_v3_change_affected,
        0 as friction_v3_change_res_known,
        0 as friction_v3_change_resolved,
        0 as friction_v3_docreq_eligible,
        0 as friction_v3_docreq_affected,
        0 as friction_v3_docreq_res_known,
        0 as friction_v3_docreq_resolved
from    ${beh.stg}.beh_call_scv c
union all
select  t.SCV_Customer_Key as SCV_Customer_Key,
        t.ActivityDate as ActivityDate,
        cast(null as int) as InLife,
        0 as prompts,
        0 as ren_cust_actions,
        0 as inlife_contacts,
        0 as friction_events,
        0 as doc_friction_events,
        0 as chases,
        0 as concessions,
        0 as failed_payments,
        0 as failed_logins,
        0 as arrears_events,
        0 as doc_requests,
        0 as sales_calls,
        0 as events_total,
        0 as online_quotes_motor,
        0 as online_quotes_home,
        0 as online_quotes_van,
        0 as renewal_requotes,
        0 as episodes,
        0 as episodes_friction,
        0 as episodes_completed,
        0 as calls_in,
        0 as calls_in_answered,
        0 as calls_in_abandoned,
        0 as calls_in_long_wait,
        0 as calls_in_wait_s,
        0 as calls_in_talk_s,
        0 as calls_out,
        0 as calls_out_talk_s,
        0 as mtas,
        1 as mta_records,
        t.IsForced as mtas_forced,
        t.IsElective as mtas_elective,
        t.CatCover as mta_cat_cover,
        t.CatDrivers as mta_cat_drivers,
        t.CatVehicle as mta_cat_vehicle,
        t.CatAddress as mta_cat_address,
        t.CatLicence as mta_cat_licence,
        t.CatUse as mta_cat_use,
        t.CatPersonal as mta_cat_personal,
        0 as loan_status_changes,
        0 as episodes_assisted,
        0 as episodes_channel_tagged,
        0 as channel_switches,
        0 as episodes_repeat_contact,
        0 as nb_sales,
        0 as nb_sales_web_assist,
        0 as doc_decisions,
        0 as doc_rejections,
        0 as doc_attempts,
        0 as doc_nonreceipt,
        0 as doc_nonreceipt_premium,
        0 as consent_actions,
        0 as consent_withdrawals,
        0 as mfq_quotes,
        0 as mfq_retrievals,
        0 as mfq_insurers,
        0 as mfq_returned,
        0 as mfq_cover_compared,
        0 as mfq_cover_down,
        0 as mfq_cover_up,
        0 as mfq_device_tagged,
        0 as mfq_device_switch,
        0 as web_sessions,
        0 as web_sessions_nonquote,
        0 as web_page_views,
        0 as web_content_views,
        0 as web_other_product_sessions,
        0 as calls_other_queues,
        0 as friction_all,
        0 as friction_resolved,
        0 as friction_payment,
        0 as friction_payment_resolved,
        0 as friction_login,
        0 as friction_login_resolved,
        0 as friction_contact,
        0 as friction_contact_resolved,
        0 as friction_documents,
        0 as friction_documents_resolved,
        0 as friction_quote,
        0 as friction_quote_resolved,
        0 as friction_process,
        0 as friction_process_resolved,
        0 as friction_other,
        0 as friction_other_resolved,
        0 as offers_sent,
        0 as offers_clicked,
        0 as offer_clicks_nb,
        0 as offer_clicks_renewal,
        0 as friction_v3_eligible,
        0 as friction_v3_affected,
        0 as friction_v3_res_known,
        0 as friction_v3_resolved,
        0 as friction_v3_renewal_eligible,
        0 as friction_v3_renewal_affected,
        0 as friction_v3_renewal_res_known,
        0 as friction_v3_renewal_resolved,
        0 as friction_v3_firstyear_eligible,
        0 as friction_v3_firstyear_affected,
        0 as friction_v3_firstyear_res_known,
        0 as friction_v3_firstyear_resolved,
        0 as friction_v3_change_eligible,
        0 as friction_v3_change_affected,
        0 as friction_v3_change_res_known,
        0 as friction_v3_change_resolved,
        0 as friction_v3_docreq_eligible,
        0 as friction_v3_docreq_affected,
        0 as friction_v3_docreq_res_known,
        0 as friction_v3_docreq_resolved
from    ${beh.stg}.beh_mta t
union all
select  q.SCV_Customer_Key as SCV_Customer_Key,
        q.ActivityDate as ActivityDate,
        cast(null as int) as InLife,
        0 as prompts,
        0 as ren_cust_actions,
        0 as inlife_contacts,
        0 as friction_events,
        0 as doc_friction_events,
        0 as chases,
        0 as concessions,
        0 as failed_payments,
        0 as failed_logins,
        0 as arrears_events,
        0 as doc_requests,
        0 as sales_calls,
        0 as events_total,
        0 as online_quotes_motor,
        0 as online_quotes_home,
        0 as online_quotes_van,
        0 as renewal_requotes,
        0 as episodes,
        0 as episodes_friction,
        0 as episodes_completed,
        0 as calls_in,
        0 as calls_in_answered,
        0 as calls_in_abandoned,
        0 as calls_in_long_wait,
        0 as calls_in_wait_s,
        0 as calls_in_talk_s,
        0 as calls_out,
        0 as calls_out_talk_s,
        q.MtaCount as mtas,
        0 as mta_records,
        0 as mtas_forced,
        0 as mtas_elective,
        0 as mta_cat_cover,
        0 as mta_cat_drivers,
        0 as mta_cat_vehicle,
        0 as mta_cat_address,
        0 as mta_cat_licence,
        0 as mta_cat_use,
        0 as mta_cat_personal,
        0 as loan_status_changes,
        0 as episodes_assisted,
        0 as episodes_channel_tagged,
        0 as channel_switches,
        0 as episodes_repeat_contact,
        0 as nb_sales,
        0 as nb_sales_web_assist,
        0 as doc_decisions,
        0 as doc_rejections,
        0 as doc_attempts,
        0 as doc_nonreceipt,
        0 as doc_nonreceipt_premium,
        0 as consent_actions,
        0 as consent_withdrawals,
        0 as mfq_quotes,
        0 as mfq_retrievals,
        0 as mfq_insurers,
        0 as mfq_returned,
        0 as mfq_cover_compared,
        0 as mfq_cover_down,
        0 as mfq_cover_up,
        0 as mfq_device_tagged,
        0 as mfq_device_switch,
        0 as web_sessions,
        0 as web_sessions_nonquote,
        0 as web_page_views,
        0 as web_content_views,
        0 as web_other_product_sessions,
        0 as calls_other_queues,
        0 as friction_all,
        0 as friction_resolved,
        0 as friction_payment,
        0 as friction_payment_resolved,
        0 as friction_login,
        0 as friction_login_resolved,
        0 as friction_contact,
        0 as friction_contact_resolved,
        0 as friction_documents,
        0 as friction_documents_resolved,
        0 as friction_quote,
        0 as friction_quote_resolved,
        0 as friction_process,
        0 as friction_process_resolved,
        0 as friction_other,
        0 as friction_other_resolved,
        0 as offers_sent,
        0 as offers_clicked,
        0 as offer_clicks_nb,
        0 as offer_clicks_renewal,
        0 as friction_v3_eligible,
        0 as friction_v3_affected,
        0 as friction_v3_res_known,
        0 as friction_v3_resolved,
        0 as friction_v3_renewal_eligible,
        0 as friction_v3_renewal_affected,
        0 as friction_v3_renewal_res_known,
        0 as friction_v3_renewal_resolved,
        0 as friction_v3_firstyear_eligible,
        0 as friction_v3_firstyear_affected,
        0 as friction_v3_firstyear_res_known,
        0 as friction_v3_firstyear_resolved,
        0 as friction_v3_change_eligible,
        0 as friction_v3_change_affected,
        0 as friction_v3_change_res_known,
        0 as friction_v3_change_resolved,
        0 as friction_v3_docreq_eligible,
        0 as friction_v3_docreq_affected,
        0 as friction_v3_docreq_res_known,
        0 as friction_v3_docreq_resolved
from    ${beh.stg}.beh_pme_mta q
union all
select  o.SCV_Customer_Key as SCV_Customer_Key,
        o.ActivityDate as ActivityDate,
        cast(null as int) as InLife,
        0 as prompts,
        0 as ren_cust_actions,
        0 as inlife_contacts,
        0 as friction_events,
        0 as doc_friction_events,
        0 as chases,
        0 as concessions,
        0 as failed_payments,
        0 as failed_logins,
        0 as arrears_events,
        0 as doc_requests,
        0 as sales_calls,
        0 as events_total,
        0 as online_quotes_motor,
        0 as online_quotes_home,
        0 as online_quotes_van,
        0 as renewal_requotes,
        0 as episodes,
        0 as episodes_friction,
        0 as episodes_completed,
        0 as calls_in,
        0 as calls_in_answered,
        0 as calls_in_abandoned,
        0 as calls_in_long_wait,
        0 as calls_in_wait_s,
        0 as calls_in_talk_s,
        0 as calls_out,
        0 as calls_out_talk_s,
        0 as mtas,
        0 as mta_records,
        0 as mtas_forced,
        0 as mtas_elective,
        0 as mta_cat_cover,
        0 as mta_cat_drivers,
        0 as mta_cat_vehicle,
        0 as mta_cat_address,
        0 as mta_cat_licence,
        0 as mta_cat_use,
        0 as mta_cat_personal,
        1 as loan_status_changes,
        0 as episodes_assisted,
        0 as episodes_channel_tagged,
        0 as channel_switches,
        0 as episodes_repeat_contact,
        0 as nb_sales,
        0 as nb_sales_web_assist,
        0 as doc_decisions,
        0 as doc_rejections,
        0 as doc_attempts,
        0 as doc_nonreceipt,
        0 as doc_nonreceipt_premium,
        0 as consent_actions,
        0 as consent_withdrawals,
        0 as mfq_quotes,
        0 as mfq_retrievals,
        0 as mfq_insurers,
        0 as mfq_returned,
        0 as mfq_cover_compared,
        0 as mfq_cover_down,
        0 as mfq_cover_up,
        0 as mfq_device_tagged,
        0 as mfq_device_switch,
        0 as web_sessions,
        0 as web_sessions_nonquote,
        0 as web_page_views,
        0 as web_content_views,
        0 as web_other_product_sessions,
        0 as calls_other_queues,
        0 as friction_all,
        0 as friction_resolved,
        0 as friction_payment,
        0 as friction_payment_resolved,
        0 as friction_login,
        0 as friction_login_resolved,
        0 as friction_contact,
        0 as friction_contact_resolved,
        0 as friction_documents,
        0 as friction_documents_resolved,
        0 as friction_quote,
        0 as friction_quote_resolved,
        0 as friction_process,
        0 as friction_process_resolved,
        0 as friction_other,
        0 as friction_other_resolved,
        0 as offers_sent,
        0 as offers_clicked,
        0 as offer_clicks_nb,
        0 as offer_clicks_renewal,
        0 as friction_v3_eligible,
        0 as friction_v3_affected,
        0 as friction_v3_res_known,
        0 as friction_v3_resolved,
        0 as friction_v3_renewal_eligible,
        0 as friction_v3_renewal_affected,
        0 as friction_v3_renewal_res_known,
        0 as friction_v3_renewal_resolved,
        0 as friction_v3_firstyear_eligible,
        0 as friction_v3_firstyear_affected,
        0 as friction_v3_firstyear_res_known,
        0 as friction_v3_firstyear_resolved,
        0 as friction_v3_change_eligible,
        0 as friction_v3_change_affected,
        0 as friction_v3_change_res_known,
        0 as friction_v3_change_resolved,
        0 as friction_v3_docreq_eligible,
        0 as friction_v3_docreq_affected,
        0 as friction_v3_docreq_res_known,
        0 as friction_v3_docreq_resolved
from    ${beh.stg}.beh_loan_change o
union all
select  f.SCV_Customer_Key as SCV_Customer_Key,
        f.ActivityDate as ActivityDate,
        cast(0 as int) as InLife,
        0 as prompts,
        0 as ren_cust_actions,
        0 as inlife_contacts,
        0 as friction_events,
        0 as doc_friction_events,
        0 as chases,
        1 as concessions,
        0 as failed_payments,
        0 as failed_logins,
        0 as arrears_events,
        0 as doc_requests,
        0 as sales_calls,
        0 as events_total,
        0 as online_quotes_motor,
        0 as online_quotes_home,
        0 as online_quotes_van,
        0 as renewal_requotes,
        0 as episodes,
        0 as episodes_friction,
        0 as episodes_completed,
        0 as calls_in,
        0 as calls_in_answered,
        0 as calls_in_abandoned,
        0 as calls_in_long_wait,
        0 as calls_in_wait_s,
        0 as calls_in_talk_s,
        0 as calls_out,
        0 as calls_out_talk_s,
        0 as mtas,
        0 as mta_records,
        0 as mtas_forced,
        0 as mtas_elective,
        0 as mta_cat_cover,
        0 as mta_cat_drivers,
        0 as mta_cat_vehicle,
        0 as mta_cat_address,
        0 as mta_cat_licence,
        0 as mta_cat_use,
        0 as mta_cat_personal,
        0 as loan_status_changes,
        0 as episodes_assisted,
        0 as episodes_channel_tagged,
        0 as channel_switches,
        0 as episodes_repeat_contact,
        0 as nb_sales,
        0 as nb_sales_web_assist,
        0 as doc_decisions,
        0 as doc_rejections,
        0 as doc_attempts,
        0 as doc_nonreceipt,
        0 as doc_nonreceipt_premium,
        0 as consent_actions,
        0 as consent_withdrawals,
        0 as mfq_quotes,
        0 as mfq_retrievals,
        0 as mfq_insurers,
        0 as mfq_returned,
        0 as mfq_cover_compared,
        0 as mfq_cover_down,
        0 as mfq_cover_up,
        0 as mfq_device_tagged,
        0 as mfq_device_switch,
        0 as web_sessions,
        0 as web_sessions_nonquote,
        0 as web_page_views,
        0 as web_content_views,
        0 as web_other_product_sessions,
        0 as calls_other_queues,
        0 as friction_all,
        0 as friction_resolved,
        0 as friction_payment,
        0 as friction_payment_resolved,
        0 as friction_login,
        0 as friction_login_resolved,
        0 as friction_contact,
        0 as friction_contact_resolved,
        0 as friction_documents,
        0 as friction_documents_resolved,
        0 as friction_quote,
        0 as friction_quote_resolved,
        0 as friction_process,
        0 as friction_process_resolved,
        0 as friction_other,
        0 as friction_other_resolved,
        0 as offers_sent,
        0 as offers_clicked,
        0 as offer_clicks_nb,
        0 as offer_clicks_renewal,
        0 as friction_v3_eligible,
        0 as friction_v3_affected,
        0 as friction_v3_res_known,
        0 as friction_v3_resolved,
        0 as friction_v3_renewal_eligible,
        0 as friction_v3_renewal_affected,
        0 as friction_v3_renewal_res_known,
        0 as friction_v3_renewal_resolved,
        0 as friction_v3_firstyear_eligible,
        0 as friction_v3_firstyear_affected,
        0 as friction_v3_firstyear_res_known,
        0 as friction_v3_firstyear_resolved,
        0 as friction_v3_change_eligible,
        0 as friction_v3_change_affected,
        0 as friction_v3_change_res_known,
        0 as friction_v3_change_resolved,
        0 as friction_v3_docreq_eligible,
        0 as friction_v3_docreq_affected,
        0 as friction_v3_docreq_res_known,
        0 as friction_v3_docreq_resolved
from    ${beh.stg}.beh_fee_concession f
union all
select  s.SCV_Customer_Key as SCV_Customer_Key,
        s.ActivityDate as ActivityDate,
        cast(null as int) as InLife,
        0 as prompts,
        0 as ren_cust_actions,
        0 as inlife_contacts,
        0 as friction_events,
        0 as doc_friction_events,
        0 as chases,
        0 as concessions,
        0 as failed_payments,
        0 as failed_logins,
        0 as arrears_events,
        0 as doc_requests,
        0 as sales_calls,
        0 as events_total,
        0 as online_quotes_motor,
        0 as online_quotes_home,
        0 as online_quotes_van,
        0 as renewal_requotes,
        0 as episodes,
        0 as episodes_friction,
        0 as episodes_completed,
        0 as calls_in,
        0 as calls_in_answered,
        0 as calls_in_abandoned,
        0 as calls_in_long_wait,
        0 as calls_in_wait_s,
        0 as calls_in_talk_s,
        0 as calls_out,
        0 as calls_out_talk_s,
        0 as mtas,
        0 as mta_records,
        0 as mtas_forced,
        0 as mtas_elective,
        0 as mta_cat_cover,
        0 as mta_cat_drivers,
        0 as mta_cat_vehicle,
        0 as mta_cat_address,
        0 as mta_cat_licence,
        0 as mta_cat_use,
        0 as mta_cat_personal,
        0 as loan_status_changes,
        0 as episodes_assisted,
        0 as episodes_channel_tagged,
        0 as channel_switches,
        0 as episodes_repeat_contact,
        1 as nb_sales,
        s.IsWebAssist as nb_sales_web_assist,
        0 as doc_decisions,
        0 as doc_rejections,
        0 as doc_attempts,
        0 as doc_nonreceipt,
        0 as doc_nonreceipt_premium,
        0 as consent_actions,
        0 as consent_withdrawals,
        0 as mfq_quotes,
        0 as mfq_retrievals,
        0 as mfq_insurers,
        0 as mfq_returned,
        0 as mfq_cover_compared,
        0 as mfq_cover_down,
        0 as mfq_cover_up,
        0 as mfq_device_tagged,
        0 as mfq_device_switch,
        0 as web_sessions,
        0 as web_sessions_nonquote,
        0 as web_page_views,
        0 as web_content_views,
        0 as web_other_product_sessions,
        0 as calls_other_queues,
        0 as friction_all,
        0 as friction_resolved,
        0 as friction_payment,
        0 as friction_payment_resolved,
        0 as friction_login,
        0 as friction_login_resolved,
        0 as friction_contact,
        0 as friction_contact_resolved,
        0 as friction_documents,
        0 as friction_documents_resolved,
        0 as friction_quote,
        0 as friction_quote_resolved,
        0 as friction_process,
        0 as friction_process_resolved,
        0 as friction_other,
        0 as friction_other_resolved,
        0 as offers_sent,
        0 as offers_clicked,
        0 as offer_clicks_nb,
        0 as offer_clicks_renewal,
        0 as friction_v3_eligible,
        0 as friction_v3_affected,
        0 as friction_v3_res_known,
        0 as friction_v3_resolved,
        0 as friction_v3_renewal_eligible,
        0 as friction_v3_renewal_affected,
        0 as friction_v3_renewal_res_known,
        0 as friction_v3_renewal_resolved,
        0 as friction_v3_firstyear_eligible,
        0 as friction_v3_firstyear_affected,
        0 as friction_v3_firstyear_res_known,
        0 as friction_v3_firstyear_resolved,
        0 as friction_v3_change_eligible,
        0 as friction_v3_change_affected,
        0 as friction_v3_change_res_known,
        0 as friction_v3_change_resolved,
        0 as friction_v3_docreq_eligible,
        0 as friction_v3_docreq_affected,
        0 as friction_v3_docreq_res_known,
        0 as friction_v3_docreq_resolved
from    ${beh.stg}.beh_sale s
union all
select  w.SCV_Customer_Key as SCV_Customer_Key,
        w.ActivityDate as ActivityDate,
        cast(null as int) as InLife,
        0 as prompts,
        0 as ren_cust_actions,
        0 as inlife_contacts,
        0 as friction_events,
        0 as doc_friction_events,
        0 as chases,
        0 as concessions,
        0 as failed_payments,
        0 as failed_logins,
        0 as arrears_events,
        0 as doc_requests,
        0 as sales_calls,
        0 as events_total,
        0 as online_quotes_motor,
        0 as online_quotes_home,
        0 as online_quotes_van,
        0 as renewal_requotes,
        0 as episodes,
        0 as episodes_friction,
        0 as episodes_completed,
        0 as calls_in,
        0 as calls_in_answered,
        0 as calls_in_abandoned,
        0 as calls_in_long_wait,
        0 as calls_in_wait_s,
        0 as calls_in_talk_s,
        0 as calls_out,
        0 as calls_out_talk_s,
        0 as mtas,
        0 as mta_records,
        0 as mtas_forced,
        0 as mtas_elective,
        0 as mta_cat_cover,
        0 as mta_cat_drivers,
        0 as mta_cat_vehicle,
        0 as mta_cat_address,
        0 as mta_cat_licence,
        0 as mta_cat_use,
        0 as mta_cat_personal,
        0 as loan_status_changes,
        0 as episodes_assisted,
        0 as episodes_channel_tagged,
        0 as channel_switches,
        0 as episodes_repeat_contact,
        0 as nb_sales,
        0 as nb_sales_web_assist,
        1 as doc_decisions,
        w.IsRejected as doc_rejections,
        w.Attempts as doc_attempts,
        0 as doc_nonreceipt,
        0 as doc_nonreceipt_premium,
        0 as consent_actions,
        0 as consent_withdrawals,
        0 as mfq_quotes,
        0 as mfq_retrievals,
        0 as mfq_insurers,
        0 as mfq_returned,
        0 as mfq_cover_compared,
        0 as mfq_cover_down,
        0 as mfq_cover_up,
        0 as mfq_device_tagged,
        0 as mfq_device_switch,
        0 as web_sessions,
        0 as web_sessions_nonquote,
        0 as web_page_views,
        0 as web_content_views,
        0 as web_other_product_sessions,
        0 as calls_other_queues,
        0 as friction_all,
        0 as friction_resolved,
        0 as friction_payment,
        0 as friction_payment_resolved,
        0 as friction_login,
        0 as friction_login_resolved,
        0 as friction_contact,
        0 as friction_contact_resolved,
        0 as friction_documents,
        0 as friction_documents_resolved,
        0 as friction_quote,
        0 as friction_quote_resolved,
        0 as friction_process,
        0 as friction_process_resolved,
        0 as friction_other,
        0 as friction_other_resolved,
        0 as offers_sent,
        0 as offers_clicked,
        0 as offer_clicks_nb,
        0 as offer_clicks_renewal,
        0 as friction_v3_eligible,
        0 as friction_v3_affected,
        0 as friction_v3_res_known,
        0 as friction_v3_resolved,
        0 as friction_v3_renewal_eligible,
        0 as friction_v3_renewal_affected,
        0 as friction_v3_renewal_res_known,
        0 as friction_v3_renewal_resolved,
        0 as friction_v3_firstyear_eligible,
        0 as friction_v3_firstyear_affected,
        0 as friction_v3_firstyear_res_known,
        0 as friction_v3_firstyear_resolved,
        0 as friction_v3_change_eligible,
        0 as friction_v3_change_affected,
        0 as friction_v3_change_res_known,
        0 as friction_v3_change_resolved,
        0 as friction_v3_docreq_eligible,
        0 as friction_v3_docreq_affected,
        0 as friction_v3_docreq_res_known,
        0 as friction_v3_docreq_resolved
from    ${beh.stg}.beh_doc_wf w
union all
select  n.SCV_Customer_Key as SCV_Customer_Key,
        n.ActivityDate as ActivityDate,
        cast(null as int) as InLife,
        0 as prompts,
        0 as ren_cust_actions,
        0 as inlife_contacts,
        0 as friction_events,
        0 as doc_friction_events,
        0 as chases,
        0 as concessions,
        0 as failed_payments,
        0 as failed_logins,
        0 as arrears_events,
        0 as doc_requests,
        0 as sales_calls,
        0 as events_total,
        0 as online_quotes_motor,
        0 as online_quotes_home,
        0 as online_quotes_van,
        0 as renewal_requotes,
        0 as episodes,
        0 as episodes_friction,
        0 as episodes_completed,
        0 as calls_in,
        0 as calls_in_answered,
        0 as calls_in_abandoned,
        0 as calls_in_long_wait,
        0 as calls_in_wait_s,
        0 as calls_in_talk_s,
        0 as calls_out,
        0 as calls_out_talk_s,
        0 as mtas,
        0 as mta_records,
        0 as mtas_forced,
        0 as mtas_elective,
        0 as mta_cat_cover,
        0 as mta_cat_drivers,
        0 as mta_cat_vehicle,
        0 as mta_cat_address,
        0 as mta_cat_licence,
        0 as mta_cat_use,
        0 as mta_cat_personal,
        0 as loan_status_changes,
        0 as episodes_assisted,
        0 as episodes_channel_tagged,
        0 as channel_switches,
        0 as episodes_repeat_contact,
        0 as nb_sales,
        0 as nb_sales_web_assist,
        0 as doc_decisions,
        0 as doc_rejections,
        0 as doc_attempts,
        0 as doc_nonreceipt,
        n.PremiumEUR as doc_nonreceipt_premium,
        0 as consent_actions,
        0 as consent_withdrawals,
        0 as mfq_quotes,
        0 as mfq_retrievals,
        0 as mfq_insurers,
        0 as mfq_returned,
        0 as mfq_cover_compared,
        0 as mfq_cover_down,
        0 as mfq_cover_up,
        0 as mfq_device_tagged,
        0 as mfq_device_switch,
        0 as web_sessions,
        0 as web_sessions_nonquote,
        0 as web_page_views,
        0 as web_content_views,
        0 as web_other_product_sessions,
        0 as calls_other_queues,
        0 as friction_all,
        0 as friction_resolved,
        0 as friction_payment,
        0 as friction_payment_resolved,
        0 as friction_login,
        0 as friction_login_resolved,
        0 as friction_contact,
        0 as friction_contact_resolved,
        0 as friction_documents,
        0 as friction_documents_resolved,
        0 as friction_quote,
        0 as friction_quote_resolved,
        0 as friction_process,
        0 as friction_process_resolved,
        0 as friction_other,
        0 as friction_other_resolved,
        0 as offers_sent,
        0 as offers_clicked,
        0 as offer_clicks_nb,
        0 as offer_clicks_renewal,
        0 as friction_v3_eligible,
        0 as friction_v3_affected,
        0 as friction_v3_res_known,
        0 as friction_v3_resolved,
        0 as friction_v3_renewal_eligible,
        0 as friction_v3_renewal_affected,
        0 as friction_v3_renewal_res_known,
        0 as friction_v3_renewal_resolved,
        0 as friction_v3_firstyear_eligible,
        0 as friction_v3_firstyear_affected,
        0 as friction_v3_firstyear_res_known,
        0 as friction_v3_firstyear_resolved,
        0 as friction_v3_change_eligible,
        0 as friction_v3_change_affected,
        0 as friction_v3_change_res_known,
        0 as friction_v3_change_resolved,
        0 as friction_v3_docreq_eligible,
        0 as friction_v3_docreq_affected,
        0 as friction_v3_docreq_res_known,
        0 as friction_v3_docreq_resolved
from    ${beh.stg}.beh_doc_nonreceipt n
union all
select  k.SCV_Customer_Key as SCV_Customer_Key,
        k.ActivityDate as ActivityDate,
        cast(null as int) as InLife,
        0 as prompts,
        0 as ren_cust_actions,
        0 as inlife_contacts,
        0 as friction_events,
        0 as doc_friction_events,
        0 as chases,
        0 as concessions,
        0 as failed_payments,
        0 as failed_logins,
        0 as arrears_events,
        0 as doc_requests,
        0 as sales_calls,
        0 as events_total,
        0 as online_quotes_motor,
        0 as online_quotes_home,
        0 as online_quotes_van,
        0 as renewal_requotes,
        0 as episodes,
        0 as episodes_friction,
        0 as episodes_completed,
        0 as calls_in,
        0 as calls_in_answered,
        0 as calls_in_abandoned,
        0 as calls_in_long_wait,
        0 as calls_in_wait_s,
        0 as calls_in_talk_s,
        0 as calls_out,
        0 as calls_out_talk_s,
        0 as mtas,
        0 as mta_records,
        0 as mtas_forced,
        0 as mtas_elective,
        0 as mta_cat_cover,
        0 as mta_cat_drivers,
        0 as mta_cat_vehicle,
        0 as mta_cat_address,
        0 as mta_cat_licence,
        0 as mta_cat_use,
        0 as mta_cat_personal,
        0 as loan_status_changes,
        0 as episodes_assisted,
        0 as episodes_channel_tagged,
        0 as channel_switches,
        0 as episodes_repeat_contact,
        0 as nb_sales,
        0 as nb_sales_web_assist,
        0 as doc_decisions,
        0 as doc_rejections,
        0 as doc_attempts,
        0 as doc_nonreceipt,
        0 as doc_nonreceipt_premium,
        1 as consent_actions,
        k.IsWithdrawal as consent_withdrawals,
        0 as mfq_quotes,
        0 as mfq_retrievals,
        0 as mfq_insurers,
        0 as mfq_returned,
        0 as mfq_cover_compared,
        0 as mfq_cover_down,
        0 as mfq_cover_up,
        0 as mfq_device_tagged,
        0 as mfq_device_switch,
        0 as web_sessions,
        0 as web_sessions_nonquote,
        0 as web_page_views,
        0 as web_content_views,
        0 as web_other_product_sessions,
        0 as calls_other_queues,
        0 as friction_all,
        0 as friction_resolved,
        0 as friction_payment,
        0 as friction_payment_resolved,
        0 as friction_login,
        0 as friction_login_resolved,
        0 as friction_contact,
        0 as friction_contact_resolved,
        0 as friction_documents,
        0 as friction_documents_resolved,
        0 as friction_quote,
        0 as friction_quote_resolved,
        0 as friction_process,
        0 as friction_process_resolved,
        0 as friction_other,
        0 as friction_other_resolved,
        0 as offers_sent,
        0 as offers_clicked,
        0 as offer_clicks_nb,
        0 as offer_clicks_renewal,
        0 as friction_v3_eligible,
        0 as friction_v3_affected,
        0 as friction_v3_res_known,
        0 as friction_v3_resolved,
        0 as friction_v3_renewal_eligible,
        0 as friction_v3_renewal_affected,
        0 as friction_v3_renewal_res_known,
        0 as friction_v3_renewal_resolved,
        0 as friction_v3_firstyear_eligible,
        0 as friction_v3_firstyear_affected,
        0 as friction_v3_firstyear_res_known,
        0 as friction_v3_firstyear_resolved,
        0 as friction_v3_change_eligible,
        0 as friction_v3_change_affected,
        0 as friction_v3_change_res_known,
        0 as friction_v3_change_resolved,
        0 as friction_v3_docreq_eligible,
        0 as friction_v3_docreq_affected,
        0 as friction_v3_docreq_res_known,
        0 as friction_v3_docreq_resolved
from    ${beh.stg}.beh_consent k
union all
select  q.SCV_Customer_Key as SCV_Customer_Key,
        q.ActivityDate as ActivityDate,
        cast(null as int) as InLife,
        0 as prompts,
        0 as ren_cust_actions,
        0 as inlife_contacts,
        0 as friction_events,
        0 as doc_friction_events,
        0 as chases,
        0 as concessions,
        0 as failed_payments,
        0 as failed_logins,
        0 as arrears_events,
        0 as doc_requests,
        0 as sales_calls,
        0 as events_total,
        0 as online_quotes_motor,
        0 as online_quotes_home,
        0 as online_quotes_van,
        0 as renewal_requotes,
        0 as episodes,
        0 as episodes_friction,
        0 as episodes_completed,
        0 as calls_in,
        0 as calls_in_answered,
        0 as calls_in_abandoned,
        0 as calls_in_long_wait,
        0 as calls_in_wait_s,
        0 as calls_in_talk_s,
        0 as calls_out,
        0 as calls_out_talk_s,
        0 as mtas,
        0 as mta_records,
        0 as mtas_forced,
        0 as mtas_elective,
        0 as mta_cat_cover,
        0 as mta_cat_drivers,
        0 as mta_cat_vehicle,
        0 as mta_cat_address,
        0 as mta_cat_licence,
        0 as mta_cat_use,
        0 as mta_cat_personal,
        0 as loan_status_changes,
        0 as episodes_assisted,
        0 as episodes_channel_tagged,
        0 as channel_switches,
        0 as episodes_repeat_contact,
        0 as nb_sales,
        0 as nb_sales_web_assist,
        0 as doc_decisions,
        0 as doc_rejections,
        0 as doc_attempts,
        0 as doc_nonreceipt,
        0 as doc_nonreceipt_premium,
        0 as consent_actions,
        0 as consent_withdrawals,
        1 as mfq_quotes,
        q.IsRetrieval as mfq_retrievals,
        q.InsurerCount as mfq_insurers,
        q.Returned as mfq_returned,
        q.CoverCompared as mfq_cover_compared,
        q.CoverDown as mfq_cover_down,
        q.CoverUp as mfq_cover_up,
        q.DeviceTagged as mfq_device_tagged,
        q.DeviceSwitch as mfq_device_switch,
        0 as web_sessions,
        0 as web_sessions_nonquote,
        0 as web_page_views,
        0 as web_content_views,
        0 as web_other_product_sessions,
        0 as calls_other_queues,
        0 as friction_all,
        0 as friction_resolved,
        0 as friction_payment,
        0 as friction_payment_resolved,
        0 as friction_login,
        0 as friction_login_resolved,
        0 as friction_contact,
        0 as friction_contact_resolved,
        0 as friction_documents,
        0 as friction_documents_resolved,
        0 as friction_quote,
        0 as friction_quote_resolved,
        0 as friction_process,
        0 as friction_process_resolved,
        0 as friction_other,
        0 as friction_other_resolved,
        0 as offers_sent,
        0 as offers_clicked,
        0 as offer_clicks_nb,
        0 as offer_clicks_renewal,
        0 as friction_v3_eligible,
        0 as friction_v3_affected,
        0 as friction_v3_res_known,
        0 as friction_v3_resolved,
        0 as friction_v3_renewal_eligible,
        0 as friction_v3_renewal_affected,
        0 as friction_v3_renewal_res_known,
        0 as friction_v3_renewal_resolved,
        0 as friction_v3_firstyear_eligible,
        0 as friction_v3_firstyear_affected,
        0 as friction_v3_firstyear_res_known,
        0 as friction_v3_firstyear_resolved,
        0 as friction_v3_change_eligible,
        0 as friction_v3_change_affected,
        0 as friction_v3_change_res_known,
        0 as friction_v3_change_resolved,
        0 as friction_v3_docreq_eligible,
        0 as friction_v3_docreq_affected,
        0 as friction_v3_docreq_res_known,
        0 as friction_v3_docreq_resolved
from    ${beh.stg}.beh_mfq_quote_x q
union all
select  g.SCV_Customer_Key as SCV_Customer_Key,
        g.ActivityDate as ActivityDate,
        cast(null as int) as InLife,
        0 as prompts,
        0 as ren_cust_actions,
        0 as inlife_contacts,
        0 as friction_events,
        0 as doc_friction_events,
        0 as chases,
        0 as concessions,
        0 as failed_payments,
        0 as failed_logins,
        0 as arrears_events,
        0 as doc_requests,
        0 as sales_calls,
        0 as events_total,
        0 as online_quotes_motor,
        0 as online_quotes_home,
        0 as online_quotes_van,
        0 as renewal_requotes,
        0 as episodes,
        0 as episodes_friction,
        0 as episodes_completed,
        0 as calls_in,
        0 as calls_in_answered,
        0 as calls_in_abandoned,
        0 as calls_in_long_wait,
        0 as calls_in_wait_s,
        0 as calls_in_talk_s,
        0 as calls_out,
        0 as calls_out_talk_s,
        0 as mtas,
        0 as mta_records,
        0 as mtas_forced,
        0 as mtas_elective,
        0 as mta_cat_cover,
        0 as mta_cat_drivers,
        0 as mta_cat_vehicle,
        0 as mta_cat_address,
        0 as mta_cat_licence,
        0 as mta_cat_use,
        0 as mta_cat_personal,
        0 as loan_status_changes,
        0 as episodes_assisted,
        0 as episodes_channel_tagged,
        0 as channel_switches,
        0 as episodes_repeat_contact,
        0 as nb_sales,
        0 as nb_sales_web_assist,
        0 as doc_decisions,
        0 as doc_rejections,
        0 as doc_attempts,
        0 as doc_nonreceipt,
        0 as doc_nonreceipt_premium,
        0 as consent_actions,
        0 as consent_withdrawals,
        0 as mfq_quotes,
        0 as mfq_retrievals,
        0 as mfq_insurers,
        0 as mfq_returned,
        0 as mfq_cover_compared,
        0 as mfq_cover_down,
        0 as mfq_cover_up,
        0 as mfq_device_tagged,
        0 as mfq_device_switch,
        1 as web_sessions,
        1 - g.HasQuoteReference as web_sessions_nonquote,
        g.PageViews as web_page_views,
        g.ContentViews as web_content_views,
        g.OtherProduct as web_other_product_sessions,
        0 as calls_other_queues,
        0 as friction_all,
        0 as friction_resolved,
        0 as friction_payment,
        0 as friction_payment_resolved,
        0 as friction_login,
        0 as friction_login_resolved,
        0 as friction_contact,
        0 as friction_contact_resolved,
        0 as friction_documents,
        0 as friction_documents_resolved,
        0 as friction_quote,
        0 as friction_quote_resolved,
        0 as friction_process,
        0 as friction_process_resolved,
        0 as friction_other,
        0 as friction_other_resolved,
        0 as offers_sent,
        0 as offers_clicked,
        0 as offer_clicks_nb,
        0 as offer_clicks_renewal,
        0 as friction_v3_eligible,
        0 as friction_v3_affected,
        0 as friction_v3_res_known,
        0 as friction_v3_resolved,
        0 as friction_v3_renewal_eligible,
        0 as friction_v3_renewal_affected,
        0 as friction_v3_renewal_res_known,
        0 as friction_v3_renewal_resolved,
        0 as friction_v3_firstyear_eligible,
        0 as friction_v3_firstyear_affected,
        0 as friction_v3_firstyear_res_known,
        0 as friction_v3_firstyear_resolved,
        0 as friction_v3_change_eligible,
        0 as friction_v3_change_affected,
        0 as friction_v3_change_res_known,
        0 as friction_v3_change_resolved,
        0 as friction_v3_docreq_eligible,
        0 as friction_v3_docreq_affected,
        0 as friction_v3_docreq_res_known,
        0 as friction_v3_docreq_resolved
from    ${beh.stg}.beh_web_session g
union all
select  t.SCV_Customer_Key as SCV_Customer_Key,
        t.ActivityDate as ActivityDate,
        cast(null as int) as InLife,
        0 as prompts,
        0 as ren_cust_actions,
        0 as inlife_contacts,
        0 as friction_events,
        0 as doc_friction_events,
        0 as chases,
        0 as concessions,
        0 as failed_payments,
        0 as failed_logins,
        0 as arrears_events,
        0 as doc_requests,
        0 as sales_calls,
        0 as events_total,
        0 as online_quotes_motor,
        0 as online_quotes_home,
        0 as online_quotes_van,
        0 as renewal_requotes,
        0 as episodes,
        0 as episodes_friction,
        0 as episodes_completed,
        0 as calls_in,
        0 as calls_in_answered,
        0 as calls_in_abandoned,
        0 as calls_in_long_wait,
        0 as calls_in_wait_s,
        0 as calls_in_talk_s,
        0 as calls_out,
        0 as calls_out_talk_s,
        0 as mtas,
        0 as mta_records,
        0 as mtas_forced,
        0 as mtas_elective,
        0 as mta_cat_cover,
        0 as mta_cat_drivers,
        0 as mta_cat_vehicle,
        0 as mta_cat_address,
        0 as mta_cat_licence,
        0 as mta_cat_use,
        0 as mta_cat_personal,
        0 as loan_status_changes,
        0 as episodes_assisted,
        0 as episodes_channel_tagged,
        0 as channel_switches,
        0 as episodes_repeat_contact,
        0 as nb_sales,
        0 as nb_sales_web_assist,
        0 as doc_decisions,
        0 as doc_rejections,
        0 as doc_attempts,
        0 as doc_nonreceipt,
        0 as doc_nonreceipt_premium,
        0 as consent_actions,
        0 as consent_withdrawals,
        0 as mfq_quotes,
        0 as mfq_retrievals,
        0 as mfq_insurers,
        0 as mfq_returned,
        0 as mfq_cover_compared,
        0 as mfq_cover_down,
        0 as mfq_cover_up,
        0 as mfq_device_tagged,
        0 as mfq_device_switch,
        0 as web_sessions,
        0 as web_sessions_nonquote,
        0 as web_page_views,
        0 as web_content_views,
        0 as web_other_product_sessions,
        1 as calls_other_queues,
        0 as friction_all,
        0 as friction_resolved,
        0 as friction_payment,
        0 as friction_payment_resolved,
        0 as friction_login,
        0 as friction_login_resolved,
        0 as friction_contact,
        0 as friction_contact_resolved,
        0 as friction_documents,
        0 as friction_documents_resolved,
        0 as friction_quote,
        0 as friction_quote_resolved,
        0 as friction_process,
        0 as friction_process_resolved,
        0 as friction_other,
        0 as friction_other_resolved,
        0 as offers_sent,
        0 as offers_clicked,
        0 as offer_clicks_nb,
        0 as offer_clicks_renewal,
        0 as friction_v3_eligible,
        0 as friction_v3_affected,
        0 as friction_v3_res_known,
        0 as friction_v3_resolved,
        0 as friction_v3_renewal_eligible,
        0 as friction_v3_renewal_affected,
        0 as friction_v3_renewal_res_known,
        0 as friction_v3_renewal_resolved,
        0 as friction_v3_firstyear_eligible,
        0 as friction_v3_firstyear_affected,
        0 as friction_v3_firstyear_res_known,
        0 as friction_v3_firstyear_resolved,
        0 as friction_v3_change_eligible,
        0 as friction_v3_change_affected,
        0 as friction_v3_change_res_known,
        0 as friction_v3_change_resolved,
        0 as friction_v3_docreq_eligible,
        0 as friction_v3_docreq_affected,
        0 as friction_v3_docreq_res_known,
        0 as friction_v3_docreq_resolved
from    ${beh.stg}.beh_tcall t where t.NoTx7 = 1
union all
select  r.SCV_Customer_Key as SCV_Customer_Key,
        r.ActivityDate as ActivityDate,
        cast(r.InLife as int) as InLife,
        0 as prompts,
        0 as ren_cust_actions,
        0 as inlife_contacts,
        0 as friction_events,
        0 as doc_friction_events,
        0 as chases,
        0 as concessions,
        0 as failed_payments,
        0 as failed_logins,
        0 as arrears_events,
        0 as doc_requests,
        0 as sales_calls,
        0 as events_total,
        0 as online_quotes_motor,
        0 as online_quotes_home,
        0 as online_quotes_van,
        0 as renewal_requotes,
        0 as episodes,
        0 as episodes_friction,
        0 as episodes_completed,
        0 as calls_in,
        0 as calls_in_answered,
        0 as calls_in_abandoned,
        0 as calls_in_long_wait,
        0 as calls_in_wait_s,
        0 as calls_in_talk_s,
        0 as calls_out,
        0 as calls_out_talk_s,
        0 as mtas,
        0 as mta_records,
        0 as mtas_forced,
        0 as mtas_elective,
        0 as mta_cat_cover,
        0 as mta_cat_drivers,
        0 as mta_cat_vehicle,
        0 as mta_cat_address,
        0 as mta_cat_licence,
        0 as mta_cat_use,
        0 as mta_cat_personal,
        0 as loan_status_changes,
        0 as episodes_assisted,
        0 as episodes_channel_tagged,
        0 as channel_switches,
        0 as episodes_repeat_contact,
        0 as nb_sales,
        0 as nb_sales_web_assist,
        0 as doc_decisions,
        0 as doc_rejections,
        0 as doc_attempts,
        0 as doc_nonreceipt,
        0 as doc_nonreceipt_premium,
        0 as consent_actions,
        0 as consent_withdrawals,
        0 as mfq_quotes,
        0 as mfq_retrievals,
        0 as mfq_insurers,
        0 as mfq_returned,
        0 as mfq_cover_compared,
        0 as mfq_cover_down,
        0 as mfq_cover_up,
        0 as mfq_device_tagged,
        0 as mfq_device_switch,
        0 as web_sessions,
        0 as web_sessions_nonquote,
        0 as web_page_views,
        0 as web_content_views,
        0 as web_other_product_sessions,
        0 as calls_other_queues,
        r.IsEvent as friction_all,
        r.IsResolved as friction_resolved,
        case when r.Family = 'payment' then r.IsEvent else 0 end as friction_payment,
        case when r.Family = 'payment' then r.IsResolved else 0 end as friction_payment_resolved,
        case when r.Family = 'login' then r.IsEvent else 0 end as friction_login,
        case when r.Family = 'login' then r.IsResolved else 0 end as friction_login_resolved,
        case when r.Family = 'contact' then r.IsEvent else 0 end as friction_contact,
        case when r.Family = 'contact' then r.IsResolved else 0 end as friction_contact_resolved,
        case when r.Family = 'documents' then r.IsEvent else 0 end as friction_documents,
        case when r.Family = 'documents' then r.IsResolved else 0 end as friction_documents_resolved,
        case when r.Family = 'quote' then r.IsEvent else 0 end as friction_quote,
        case when r.Family = 'quote' then r.IsResolved else 0 end as friction_quote_resolved,
        case when r.Family = 'process' then r.IsEvent else 0 end as friction_process,
        case when r.Family = 'process' then r.IsResolved else 0 end as friction_process_resolved,
        case when r.Family = 'other' then r.IsEvent else 0 end as friction_other,
        case when r.Family = 'other' then r.IsResolved else 0 end as friction_other_resolved,
        0 as offers_sent,
        0 as offers_clicked,
        0 as offer_clicks_nb,
        0 as offer_clicks_renewal,
        0 as friction_v3_eligible,
        0 as friction_v3_affected,
        0 as friction_v3_res_known,
        0 as friction_v3_resolved,
        0 as friction_v3_renewal_eligible,
        0 as friction_v3_renewal_affected,
        0 as friction_v3_renewal_res_known,
        0 as friction_v3_renewal_resolved,
        0 as friction_v3_firstyear_eligible,
        0 as friction_v3_firstyear_affected,
        0 as friction_v3_firstyear_res_known,
        0 as friction_v3_firstyear_resolved,
        0 as friction_v3_change_eligible,
        0 as friction_v3_change_affected,
        0 as friction_v3_change_res_known,
        0 as friction_v3_change_resolved,
        0 as friction_v3_docreq_eligible,
        0 as friction_v3_docreq_affected,
        0 as friction_v3_docreq_res_known,
        0 as friction_v3_docreq_resolved
from    ${beh.stg}.beh_friction_res r
union all
select  o.SCV_Customer_Key as SCV_Customer_Key,
        o.ActivityDate as ActivityDate,
        cast(null as int) as InLife,
        0 as prompts,
        0 as ren_cust_actions,
        0 as inlife_contacts,
        0 as friction_events,
        0 as doc_friction_events,
        0 as chases,
        0 as concessions,
        0 as failed_payments,
        0 as failed_logins,
        0 as arrears_events,
        0 as doc_requests,
        0 as sales_calls,
        0 as events_total,
        0 as online_quotes_motor,
        0 as online_quotes_home,
        0 as online_quotes_van,
        0 as renewal_requotes,
        0 as episodes,
        0 as episodes_friction,
        0 as episodes_completed,
        0 as calls_in,
        0 as calls_in_answered,
        0 as calls_in_abandoned,
        0 as calls_in_long_wait,
        0 as calls_in_wait_s,
        0 as calls_in_talk_s,
        0 as calls_out,
        0 as calls_out_talk_s,
        0 as mtas,
        0 as mta_records,
        0 as mtas_forced,
        0 as mtas_elective,
        0 as mta_cat_cover,
        0 as mta_cat_drivers,
        0 as mta_cat_vehicle,
        0 as mta_cat_address,
        0 as mta_cat_licence,
        0 as mta_cat_use,
        0 as mta_cat_personal,
        0 as loan_status_changes,
        0 as episodes_assisted,
        0 as episodes_channel_tagged,
        0 as channel_switches,
        0 as episodes_repeat_contact,
        0 as nb_sales,
        0 as nb_sales_web_assist,
        0 as doc_decisions,
        0 as doc_rejections,
        0 as doc_attempts,
        0 as doc_nonreceipt,
        0 as doc_nonreceipt_premium,
        0 as consent_actions,
        0 as consent_withdrawals,
        0 as mfq_quotes,
        0 as mfq_retrievals,
        0 as mfq_insurers,
        0 as mfq_returned,
        0 as mfq_cover_compared,
        0 as mfq_cover_down,
        0 as mfq_cover_up,
        0 as mfq_device_tagged,
        0 as mfq_device_switch,
        0 as web_sessions,
        0 as web_sessions_nonquote,
        0 as web_page_views,
        0 as web_content_views,
        0 as web_other_product_sessions,
        0 as calls_other_queues,
        0 as friction_all,
        0 as friction_resolved,
        0 as friction_payment,
        0 as friction_payment_resolved,
        0 as friction_login,
        0 as friction_login_resolved,
        0 as friction_contact,
        0 as friction_contact_resolved,
        0 as friction_documents,
        0 as friction_documents_resolved,
        0 as friction_quote,
        0 as friction_quote_resolved,
        0 as friction_process,
        0 as friction_process_resolved,
        0 as friction_other,
        0 as friction_other_resolved,
        o.IsSend as offers_sent,
        o.IsClickedSend as offers_clicked,
        o.IsNbFollow as offer_clicks_nb,
        o.IsRenewalFollow as offer_clicks_renewal,
        0 as friction_v3_eligible,
        0 as friction_v3_affected,
        0 as friction_v3_res_known,
        0 as friction_v3_resolved,
        0 as friction_v3_renewal_eligible,
        0 as friction_v3_renewal_affected,
        0 as friction_v3_renewal_res_known,
        0 as friction_v3_renewal_resolved,
        0 as friction_v3_firstyear_eligible,
        0 as friction_v3_firstyear_affected,
        0 as friction_v3_firstyear_res_known,
        0 as friction_v3_firstyear_resolved,
        0 as friction_v3_change_eligible,
        0 as friction_v3_change_affected,
        0 as friction_v3_change_res_known,
        0 as friction_v3_change_resolved,
        0 as friction_v3_docreq_eligible,
        0 as friction_v3_docreq_affected,
        0 as friction_v3_docreq_res_known,
        0 as friction_v3_docreq_resolved
from    ${beh.stg}.beh_offer_row o
union all
select  v.SCV_Customer_Key as SCV_Customer_Key,
        v.ActivityDate as ActivityDate,
        cast(v.InLife as int) as InLife,
        0 as prompts,
        0 as ren_cust_actions,
        0 as inlife_contacts,
        0 as friction_events,
        0 as doc_friction_events,
        0 as chases,
        0 as concessions,
        0 as failed_payments,
        0 as failed_logins,
        0 as arrears_events,
        0 as doc_requests,
        0 as sales_calls,
        0 as events_total,
        0 as online_quotes_motor,
        0 as online_quotes_home,
        0 as online_quotes_van,
        0 as renewal_requotes,
        0 as episodes,
        0 as episodes_friction,
        0 as episodes_completed,
        0 as calls_in,
        0 as calls_in_answered,
        0 as calls_in_abandoned,
        0 as calls_in_long_wait,
        0 as calls_in_wait_s,
        0 as calls_in_talk_s,
        0 as calls_out,
        0 as calls_out_talk_s,
        0 as mtas,
        0 as mta_records,
        0 as mtas_forced,
        0 as mtas_elective,
        0 as mta_cat_cover,
        0 as mta_cat_drivers,
        0 as mta_cat_vehicle,
        0 as mta_cat_address,
        0 as mta_cat_licence,
        0 as mta_cat_use,
        0 as mta_cat_personal,
        0 as loan_status_changes,
        0 as episodes_assisted,
        0 as episodes_channel_tagged,
        0 as channel_switches,
        0 as episodes_repeat_contact,
        0 as nb_sales,
        0 as nb_sales_web_assist,
        0 as doc_decisions,
        0 as doc_rejections,
        0 as doc_attempts,
        0 as doc_nonreceipt,
        0 as doc_nonreceipt_premium,
        0 as consent_actions,
        0 as consent_withdrawals,
        0 as mfq_quotes,
        0 as mfq_retrievals,
        0 as mfq_insurers,
        0 as mfq_returned,
        0 as mfq_cover_compared,
        0 as mfq_cover_down,
        0 as mfq_cover_up,
        0 as mfq_device_tagged,
        0 as mfq_device_switch,
        0 as web_sessions,
        0 as web_sessions_nonquote,
        0 as web_page_views,
        0 as web_content_views,
        0 as web_other_product_sessions,
        0 as calls_other_queues,
        0 as friction_all,
        0 as friction_resolved,
        0 as friction_payment,
        0 as friction_payment_resolved,
        0 as friction_login,
        0 as friction_login_resolved,
        0 as friction_contact,
        0 as friction_contact_resolved,
        0 as friction_documents,
        0 as friction_documents_resolved,
        0 as friction_quote,
        0 as friction_quote_resolved,
        0 as friction_process,
        0 as friction_process_resolved,
        0 as friction_other,
        0 as friction_other_resolved,
        0 as offers_sent,
        0 as offers_clicked,
        0 as offer_clicks_nb,
        0 as offer_clicks_renewal,
        v.Eligible as friction_v3_eligible,
        v.Affected as friction_v3_affected,
        v.ResKnown as friction_v3_res_known,
        v.Resolved as friction_v3_resolved,
        case when v.EpisodeGroup = 'renewal' then v.Eligible else 0 end as friction_v3_renewal_eligible,
        case when v.EpisodeGroup = 'renewal' then v.Affected else 0 end as friction_v3_renewal_affected,
        case when v.EpisodeGroup = 'renewal' then v.ResKnown else 0 end as friction_v3_renewal_res_known,
        case when v.EpisodeGroup = 'renewal' then v.Resolved else 0 end as friction_v3_renewal_resolved,
        case when v.EpisodeGroup = 'firstyear' then v.Eligible else 0 end as friction_v3_firstyear_eligible,
        case when v.EpisodeGroup = 'firstyear' then v.Affected else 0 end as friction_v3_firstyear_affected,
        case when v.EpisodeGroup = 'firstyear' then v.ResKnown else 0 end as friction_v3_firstyear_res_known,
        case when v.EpisodeGroup = 'firstyear' then v.Resolved else 0 end as friction_v3_firstyear_resolved,
        case when v.EpisodeGroup = 'change' then v.Eligible else 0 end as friction_v3_change_eligible,
        case when v.EpisodeGroup = 'change' then v.Affected else 0 end as friction_v3_change_affected,
        case when v.EpisodeGroup = 'change' then v.ResKnown else 0 end as friction_v3_change_res_known,
        case when v.EpisodeGroup = 'change' then v.Resolved else 0 end as friction_v3_change_resolved,
        case when v.EpisodeGroup = 'docreq' then v.Eligible else 0 end as friction_v3_docreq_eligible,
        case when v.EpisodeGroup = 'docreq' then v.Affected else 0 end as friction_v3_docreq_affected,
        case when v.EpisodeGroup = 'docreq' then v.ResKnown else 0 end as friction_v3_docreq_res_known,
        case when v.EpisodeGroup = 'docreq' then v.Resolved else 0 end as friction_v3_docreq_resolved
from    ${beh.stg}.beh_friction_v3 v


## beh_rw_day


In [ ]:
/****************************************************************************************************/
/*  Name        : stg.beh_rw_day                                                                    */
/*  Description : Customer activity days that fall inside one of the customer's renewal windows:    */
/*                an offered Motor, Home or Van renewal's due date minus 60 to plus 30 days         */
/*                (clarifications A3). Activity on other days is in-life (_il).                     */
/*  Reads       : stg.beh_activity_row, stg.beh_renewal_due                                         */
/*  Key         : SCV_Customer_Key + ActivityDate                                                   */
/****************************************************************************************************/
create or replace table ${beh.stg}.beh_rw_day as
select  distinct a.SCV_Customer_Key,
        a.ActivityDate
from    (select distinct SCV_Customer_Key, ActivityDate from ${beh.stg}.beh_activity_row) a
join    ${beh.stg}.beh_renewal_due r
        on  r.SCV_Customer_Key = a.SCV_Customer_Key
        and r.PolicyOfferNum = 1
        and a.ActivityDate between date_sub(r.DueDate, 60) and date_add(r.DueDate, 30)


## beh_activity_month


In [ ]:
/****************************************************************************************************/
/*  Name        : stg.beh_activity_month                                                            */
/*  Description : Activity per customer and calendar month: every row measure, plus _il (outside    */
/*                renewal windows) for the measures that have one.                                  */
/*  Reads       : stg.beh_activity_row, stg.beh_rw_day                                              */
/*  Key         : SCV_Customer_Key + MonthNo                                                        */
/****************************************************************************************************/
create or replace table ${beh.stg}.beh_activity_month as
select  a.SCV_Customer_Key,
        year(a.ActivityDate) * 12 + month(a.ActivityDate) as MonthNo,
        sum(a.prompts) as prompts,
        sum(a.ren_cust_actions) as ren_cust_actions,
        sum(a.inlife_contacts) as inlife_contacts,
        sum(a.friction_events) as friction_events,
        sum(case when coalesce(a.InLife, case when w.SCV_Customer_Key is null then 1 else 0 end) = 1 then a.friction_events else 0 end) as friction_events_il,
        sum(a.doc_friction_events) as doc_friction_events,
        sum(a.chases) as chases,
        sum(case when coalesce(a.InLife, case when w.SCV_Customer_Key is null then 1 else 0 end) = 1 then a.chases else 0 end) as chases_il,
        sum(a.concessions) as concessions,
        sum(a.failed_payments) as failed_payments,
        sum(case when coalesce(a.InLife, case when w.SCV_Customer_Key is null then 1 else 0 end) = 1 then a.failed_payments else 0 end) as failed_payments_il,
        sum(a.failed_logins) as failed_logins,
        sum(a.arrears_events) as arrears_events,
        sum(a.doc_requests) as doc_requests,
        sum(a.sales_calls) as sales_calls,
        sum(case when coalesce(a.InLife, case when w.SCV_Customer_Key is null then 1 else 0 end) = 1 then a.sales_calls else 0 end) as sales_calls_il,
        sum(a.events_total) as events_total,
        sum(a.online_quotes_motor) as online_quotes_motor,
        sum(case when coalesce(a.InLife, case when w.SCV_Customer_Key is null then 1 else 0 end) = 1 then a.online_quotes_motor else 0 end) as online_quotes_motor_il,
        sum(a.online_quotes_home) as online_quotes_home,
        sum(case when coalesce(a.InLife, case when w.SCV_Customer_Key is null then 1 else 0 end) = 1 then a.online_quotes_home else 0 end) as online_quotes_home_il,
        sum(a.online_quotes_van) as online_quotes_van,
        sum(case when coalesce(a.InLife, case when w.SCV_Customer_Key is null then 1 else 0 end) = 1 then a.online_quotes_van else 0 end) as online_quotes_van_il,
        sum(a.renewal_requotes) as renewal_requotes,
        sum(a.episodes) as episodes,
        sum(case when coalesce(a.InLife, case when w.SCV_Customer_Key is null then 1 else 0 end) = 1 then a.episodes else 0 end) as episodes_il,
        sum(a.episodes_friction) as episodes_friction,
        sum(case when coalesce(a.InLife, case when w.SCV_Customer_Key is null then 1 else 0 end) = 1 then a.episodes_friction else 0 end) as episodes_friction_il,
        sum(a.episodes_completed) as episodes_completed,
        sum(a.calls_in) as calls_in,
        sum(case when coalesce(a.InLife, case when w.SCV_Customer_Key is null then 1 else 0 end) = 1 then a.calls_in else 0 end) as calls_in_il,
        sum(a.calls_in_answered) as calls_in_answered,
        sum(a.calls_in_abandoned) as calls_in_abandoned,
        sum(a.calls_in_long_wait) as calls_in_long_wait,
        sum(a.calls_in_wait_s) as calls_in_wait_s,
        sum(a.calls_in_talk_s) as calls_in_talk_s,
        sum(a.calls_out) as calls_out,
        sum(case when coalesce(a.InLife, case when w.SCV_Customer_Key is null then 1 else 0 end) = 1 then a.calls_out else 0 end) as calls_out_il,
        sum(a.calls_out_talk_s) as calls_out_talk_s,
        sum(a.mtas) as mtas,
        sum(case when coalesce(a.InLife, case when w.SCV_Customer_Key is null then 1 else 0 end) = 1 then a.mtas else 0 end) as mtas_il,
        sum(a.mta_records) as mta_records,
        sum(a.mtas_forced) as mtas_forced,
        sum(a.mtas_elective) as mtas_elective,
        sum(a.mta_cat_cover) as mta_cat_cover,
        sum(a.mta_cat_drivers) as mta_cat_drivers,
        sum(a.mta_cat_vehicle) as mta_cat_vehicle,
        sum(a.mta_cat_address) as mta_cat_address,
        sum(a.mta_cat_licence) as mta_cat_licence,
        sum(a.mta_cat_use) as mta_cat_use,
        sum(a.mta_cat_personal) as mta_cat_personal,
        sum(a.loan_status_changes) as loan_status_changes,
        sum(a.episodes_assisted) as episodes_assisted,
        sum(a.episodes_channel_tagged) as episodes_channel_tagged,
        sum(a.channel_switches) as channel_switches,
        sum(a.episodes_repeat_contact) as episodes_repeat_contact,
        sum(a.nb_sales) as nb_sales,
        sum(case when coalesce(a.InLife, case when w.SCV_Customer_Key is null then 1 else 0 end) = 1 then a.nb_sales else 0 end) as nb_sales_il,
        sum(a.nb_sales_web_assist) as nb_sales_web_assist,
        sum(case when coalesce(a.InLife, case when w.SCV_Customer_Key is null then 1 else 0 end) = 1 then a.nb_sales_web_assist else 0 end) as nb_sales_web_assist_il,
        sum(a.doc_decisions) as doc_decisions,
        sum(a.doc_rejections) as doc_rejections,
        sum(a.doc_attempts) as doc_attempts,
        sum(a.doc_nonreceipt) as doc_nonreceipt,
        sum(case when coalesce(a.InLife, case when w.SCV_Customer_Key is null then 1 else 0 end) = 1 then a.doc_nonreceipt else 0 end) as doc_nonreceipt_il,
        sum(a.doc_nonreceipt_premium) as doc_nonreceipt_premium,
        sum(a.consent_actions) as consent_actions,
        sum(case when coalesce(a.InLife, case when w.SCV_Customer_Key is null then 1 else 0 end) = 1 then a.consent_actions else 0 end) as consent_actions_il,
        sum(a.consent_withdrawals) as consent_withdrawals,
        sum(case when coalesce(a.InLife, case when w.SCV_Customer_Key is null then 1 else 0 end) = 1 then a.consent_withdrawals else 0 end) as consent_withdrawals_il,
        sum(a.mfq_quotes) as mfq_quotes,
        sum(a.mfq_retrievals) as mfq_retrievals,
        sum(a.mfq_insurers) as mfq_insurers,
        sum(a.mfq_returned) as mfq_returned,
        sum(a.mfq_cover_compared) as mfq_cover_compared,
        sum(a.mfq_cover_down) as mfq_cover_down,
        sum(a.mfq_cover_up) as mfq_cover_up,
        sum(a.mfq_device_tagged) as mfq_device_tagged,
        sum(a.mfq_device_switch) as mfq_device_switch,
        sum(a.web_sessions) as web_sessions,
        sum(a.web_sessions_nonquote) as web_sessions_nonquote,
        sum(a.web_page_views) as web_page_views,
        sum(a.web_content_views) as web_content_views,
        sum(a.web_other_product_sessions) as web_other_product_sessions,
        sum(a.calls_other_queues) as calls_other_queues,
        sum(a.friction_all) as friction_all,
        sum(a.friction_resolved) as friction_resolved,
        sum(a.friction_payment) as friction_payment,
        sum(a.friction_payment_resolved) as friction_payment_resolved,
        sum(a.friction_login) as friction_login,
        sum(a.friction_login_resolved) as friction_login_resolved,
        sum(a.friction_contact) as friction_contact,
        sum(a.friction_contact_resolved) as friction_contact_resolved,
        sum(a.friction_documents) as friction_documents,
        sum(a.friction_documents_resolved) as friction_documents_resolved,
        sum(a.friction_quote) as friction_quote,
        sum(a.friction_quote_resolved) as friction_quote_resolved,
        sum(a.friction_process) as friction_process,
        sum(a.friction_process_resolved) as friction_process_resolved,
        sum(a.friction_other) as friction_other,
        sum(a.friction_other_resolved) as friction_other_resolved,
        sum(a.offers_sent) as offers_sent,
        sum(a.offers_clicked) as offers_clicked,
        sum(a.offer_clicks_nb) as offer_clicks_nb,
        sum(a.offer_clicks_renewal) as offer_clicks_renewal,
        sum(a.friction_v3_eligible) as friction_v3_eligible,
        sum(a.friction_v3_affected) as friction_v3_affected,
        sum(a.friction_v3_res_known) as friction_v3_res_known,
        sum(a.friction_v3_resolved) as friction_v3_resolved,
        sum(a.friction_v3_renewal_eligible) as friction_v3_renewal_eligible,
        sum(a.friction_v3_renewal_affected) as friction_v3_renewal_affected,
        sum(a.friction_v3_renewal_res_known) as friction_v3_renewal_res_known,
        sum(a.friction_v3_renewal_resolved) as friction_v3_renewal_resolved,
        sum(a.friction_v3_firstyear_eligible) as friction_v3_firstyear_eligible,
        sum(a.friction_v3_firstyear_affected) as friction_v3_firstyear_affected,
        sum(a.friction_v3_firstyear_res_known) as friction_v3_firstyear_res_known,
        sum(a.friction_v3_firstyear_resolved) as friction_v3_firstyear_resolved,
        sum(a.friction_v3_change_eligible) as friction_v3_change_eligible,
        sum(a.friction_v3_change_affected) as friction_v3_change_affected,
        sum(a.friction_v3_change_res_known) as friction_v3_change_res_known,
        sum(a.friction_v3_change_resolved) as friction_v3_change_resolved,
        sum(a.friction_v3_docreq_eligible) as friction_v3_docreq_eligible,
        sum(a.friction_v3_docreq_affected) as friction_v3_docreq_affected,
        sum(a.friction_v3_docreq_res_known) as friction_v3_docreq_res_known,
        sum(a.friction_v3_docreq_resolved) as friction_v3_docreq_resolved
from    ${beh.stg}.beh_activity_row a
left join ${beh.stg}.beh_rw_day w
        on  w.SCV_Customer_Key = a.SCV_Customer_Key
        and w.ActivityDate = a.ActivityDate
where   a.ActivityDate <= to_date('${beh.last_me}')
group by a.SCV_Customer_Key, year(a.ActivityDate) * 12 + month(a.ActivityDate)


## beh_comms_msg


In [ ]:
/****************************************************************************************************/
/*  Name        : stg.beh_comms_msg                                                                 */
/*  Description : CRM messages sent (campaign history rows with a send), with the send date and     */
/*                the open, click and bounce flags. Opens and clicks are flags per message, never   */
/*                open counts (OpenCount counts events).                                            */
/*  Reads       : ads_v2.CampaignHistory                                                            */
/*  Key         : CH_Id                                                                             */
/****************************************************************************************************/
create or replace table ${beh.stg}.beh_comms_msg as
select  CH_Id,
        SCV_Customer_Key,
        coalesce(to_date(min_SentDateTime),
                 to_date(concat(cast(MonthId as string), '01'), 'yyyyMMdd'))          as SendDate,
        cast(floor(MonthId / 100) as int) * 12 + MonthId % 100                       as MonthNo,
        case when OpenCount > 0 then 1 else 0 end                                    as Opened,
        case when ClickCount > 0 then 1 else 0 end                                   as Clicked,
        case when Bounced = 1 then 1 else 0 end                                      as Bounced
from    ${beh.ads}.campaignhistory
where   SCV_Customer_Key is not null
  and   SentCount > 0
  and   MonthId >= cast(date_format(to_date('${beh.cov_comms}'), 'yyyyMM') as int)
  and   MonthId <= cast(date_format(to_date('${beh.last_me}'), 'yyyyMM') as int)


## beh_comms_rw_day


In [ ]:
/****************************************************************************************************/
/*  Name        : stg.beh_comms_rw_day                                                              */
/*  Description : Customer send days inside one of the customer's renewal windows (offered renewal  */
/*                due date minus 60 to plus 30 days). Messages on other days are in-life (_il).     */
/*  Reads       : stg.beh_comms_msg, stg.beh_renewal_due                                            */
/*  Key         : SCV_Customer_Key + SendDate                                                       */
/****************************************************************************************************/
create or replace table ${beh.stg}.beh_comms_rw_day as
select  distinct m.SCV_Customer_Key,
        m.SendDate
from    (select distinct SCV_Customer_Key, SendDate from ${beh.stg}.beh_comms_msg) m
join    ${beh.stg}.beh_renewal_due r
        on  r.SCV_Customer_Key = m.SCV_Customer_Key
        and r.PolicyOfferNum = 1
        and m.SendDate between date_sub(r.DueDate, 60) and date_add(r.DueDate, 30)


## beh_comms_month


In [ ]:
/****************************************************************************************************/
/*  Name        : stg.beh_comms_month                                                               */
/*  Description : CRM messages per customer and send month: sent, opened, clicked, bounced, and     */
/*                the in-life (_il) sent, opened and clicked (sent outside the customer's renewal   */
/*                windows).                                                                         */
/*  Reads       : stg.beh_comms_msg, stg.beh_comms_rw_day                                           */
/*  Key         : SCV_Customer_Key + MonthNo                                                        */
/****************************************************************************************************/
create or replace table ${beh.stg}.beh_comms_month as
select  m.SCV_Customer_Key,
        m.MonthNo,
        count(*)                                                            as msgs_sent,
        sum(case when w.SCV_Customer_Key is null then 1 else 0 end)         as msgs_sent_il,
        sum(m.Opened)                                                       as msgs_opened,
        sum(case when w.SCV_Customer_Key is null then m.Opened else 0 end)  as msgs_opened_il,
        sum(m.Clicked)                                                      as msgs_clicked,
        sum(case when w.SCV_Customer_Key is null then m.Clicked else 0 end) as msgs_clicked_il,
        sum(m.Bounced)                                                      as msgs_bounced
from    ${beh.stg}.beh_comms_msg m
left join ${beh.stg}.beh_comms_rw_day w
        on  w.SCV_Customer_Key = m.SCV_Customer_Key
        and w.SendDate = m.SendDate
group by m.SCV_Customer_Key, m.MonthNo


## Checks


In [ ]:
%%pyspark
# Row count, duplicate keys and null keys for every table this notebook built. Logged to BehBuildLog;
# the cell fails if any check failed, naming the table and the likely cause.
beh_check("BEH_20_Activity", [{"table": "{stg}.beh_renewal_due", "key": ["PolicyCode", "DueDate"]}, {"table": "{stg}.beh_event", "key": ["EventId"]}, {"table": "{stg}.beh_episode", "key": ["EpisodeKey"]}, {"table": "{stg}.beh_call", "key": ["ConversationId"]}, {"table": "{stg}.beh_call_scv", "key": ["ConversationId"]}, {"table": "{stg}.beh_event_channel", "key": ["EventId"]}, {"table": "{stg}.beh_episode_call", "key": ["ConversationId"]}, {"table": "{stg}.beh_episode_x", "key": ["EpisodeKey"]}, {"table": "{stg}.beh_mta", "key": ["PolicyCode", "HistorySeq"]}, {"table": "{stg}.beh_pme_mta", "key": ["SCV_Customer_Key", "PolicyCode", "ActivityDate"]}, {"table": "{stg}.beh_loan_change", "key": ["LoanReference", "ActivityDate"]}, {"table": "{stg}.beh_fee_concession", "key": ["PolicyCode", "LYPolicyRenewDateAdj"]}, {"table": "{stg}.beh_sale", "key": ["PolicyCode"]}, {"table": "{stg}.beh_doc_wf", "key": ["Pk"]}, {"table": "{stg}.beh_doc_nonreceipt", "key": ["SCV_Customer_Key", "PolicyCode", "MonthNo"]}, {"table": "{stg}.beh_consent", "key": ["ActionTrackId"]}, {"table": "{stg}.beh_portal_month", "key": ["SCV_Customer_Key", "MonthNo"]}, {"table": "{stg}.beh_mfq_seed", "key": ["SCV_Customer_Key", "GuidKey"]}, {"table": "{stg}.beh_mfq_retrieval", "key": ["GuidKey", "RetrievedGuidKey"]}, {"table": "{stg}.beh_mfq_chain1", "key": ["SCV_Customer_Key", "GuidKey"]}, {"table": "{stg}.beh_mfq_chain2", "key": ["SCV_Customer_Key", "GuidKey"]}, {"table": "{stg}.beh_mfq_quote", "key": ["GuidKey"]}, {"table": "{stg}.beh_mfq_quote_x", "key": ["GuidKey"]}, {"table": "{stg}.beh_web_cookie", "key": ["ClientID"]}, {"table": "{stg}.beh_web_session_email", "key": ["ClientID", "SessionID"]}, {"table": "{stg}.beh_web_session", "key": ["ClientID", "SessionID"]}, {"table": "{stg}.beh_client_scv", "key": ["ClientCode"]}, {"table": "{stg}.beh_tx_day", "key": ["SCV_Customer_Key", "TxDate"]}, {"table": "{stg}.beh_tcall_base", "key": ["CallIDKey"]}, {"table": "{stg}.beh_tcall", "key": ["CallIDKey"]}, {"table": "{stg}.beh_offer_msg", "key": ["CH_Id"]}, {"table": "{stg}.beh_friction_v3", "key": ["EpisodeID"]}, {"table": "{stg}.beh_rw_day", "key": ["SCV_Customer_Key", "ActivityDate"]}, {"table": "{stg}.beh_activity_month", "key": ["SCV_Customer_Key", "MonthNo"]}, {"table": "{stg}.beh_comms_msg", "key": ["CH_Id"]}, {"table": "{stg}.beh_comms_rw_day", "key": ["SCV_Customer_Key", "SendDate"]}, {"table": "{stg}.beh_comms_month", "key": ["SCV_Customer_Key", "MonthNo"]}])


In [ ]:
%%pyspark
beh_run_log("BEH_20_Activity", "succeeded")
